In [1]:
"""
Download OHLCV data for the full ticker list and save each to its own CSV,
then zip all CSVs together.
"""

import os
import zipfile
import yfinance as yf
import pandas as pd
# ─────────────────────────────────────────────────────────────────────────────
# PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
TICKERS = [
    # US Large Cap Tech
    "AAPL", "MSFT", "GOOG", "AMZN", "NVDA",
    "META", "TSLA", "NFLX", "AMD", "INTC",

    # US Financials
    "JPM", "BAC", "GS", "MS", "WFC",
    "BLK", "C", "AXP", "V", "MA",

    # US Healthcare & Consumer
    "JNJ", "PFE", "UNH", "MRK", "ABBV",
    "PG", "KO", "PEP", "WMT", "COST",

    # US Energy & Industrials
    "XOM", "CVX", "COP", "BA", "CAT",
    "GE", "HON", "LMT", "RTX", "UPS",

    # Commodities & Crypto (yfinance format)
    "GC=F",    # Gold
    "SI=F",    # Silver
    "CL=F",    # Crude Oil
    "BTC-USD", # Bitcoin
    "ETH-USD", # Ethereum

    # International ETFs (broad exposure)
    "EWJ",  # Japan
    "EWZ",  # Brazil
    "FXI",  # China
    "EWG",  # Germany
    "EWU",  # UK
]

START = "2010-01-01"
END   = "2026-04-11"

OUT_DIR = "stock_csv_data"
os.makedirs(OUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# DOWNLOAD + SAVE EACH TICKER TO ITS OWN CSV
# ─────────────────────────────────────────────────────────────────────────────
saved_files = []

for ticker in TICKERS:
    print(f"Downloading {ticker} ...")
    try:
        data = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=True)

        # Flatten MultiIndex columns (yfinance returns ['Close','AMD'] style tuples)
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
    except Exception as e:
        print(f"  Failed to download {ticker}: {e}")
        continue

    if data is None or data.empty:
        print(f"  No data returned for {ticker}, skipping.")
        continue

    csv_name = f"{OUT_DIR}/{ticker.replace('=', '_')}_data.csv"
    data.to_csv(csv_name)
    saved_files.append(csv_name)
    print(f"  saved → {csv_name}  ({len(data)} rows)")

# ─────────────────────────────────────────────────────────────────────────────
# ZIP ALL CSVs TOGETHER
# ─────────────────────────────────────────────────────────────────────────────
zip_path = "stock_data.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fpath in saved_files:
        zf.write(fpath, arcname=os.path.basename(fpath))

print(f"\nDone. {len(saved_files)} CSVs zipped → {zip_path}")

  saved → stock_csv_data/AAPL_data.csv  (4092 rows)
  saved → stock_csv_data/MSFT_data.csv  (4092 rows)
  saved → stock_csv_data/GOOG_data.csv  (4092 rows)
  saved → stock_csv_data/AMZN_data.csv  (4092 rows)
  saved → stock_csv_data/NVDA_data.csv  (4092 rows)
  saved → stock_csv_data/META_data.csv  (3493 rows)
  saved → stock_csv_data/TSLA_data.csv  (3970 rows)
  saved → stock_csv_data/NFLX_data.csv  (4092 rows)
  saved → stock_csv_data/AMD_data.csv  (4092 rows)
  saved → stock_csv_data/INTC_data.csv  (4092 rows)
  saved → stock_csv_data/JPM_data.csv  (4092 rows)
  saved → stock_csv_data/BAC_data.csv  (4092 rows)
  saved → stock_csv_data/GS_data.csv  (4092 rows)
  saved → stock_csv_data/MS_data.csv  (4092 rows)
  saved → stock_csv_data/WFC_data.csv  (4092 rows)
  saved → stock_csv_data/BLK_data.csv  (4092 rows)
  saved → stock_csv_data/C_data.csv  (4092 rows)
  saved → stock_csv_data/AXP_data.csv  (4092 rows)
  saved → stock_csv_data/V_data.csv  (4092 rows)
  saved → stock_csv_data/MA_

In [2]:
"""
MF-DFA on Log-Returns | MF-DMA variants on Log-Prices
=======================================================================
Correct input series per method:
  - MF-DFA  → Log-Returns  (stationary, no trend contamination)
  - MF-BDMA → Log-Prices   (DMA designed for price-level series with trend)
  - MF-CDMA → Log-Prices
  - MF-FDMA → Log-Prices

DMA operates directly on log-prices (no integration step needed since
log-prices already have the slow-moving trend the moving average tracks).

DFA integrates returns first (standard DFA profile construction).

6-panel layout:
  (a) raw series         (b) F_q(s) log-log     (c) H(q)
  (d) tau(q)             (e) alpha(q)            (f) f(alpha) spectrum
"""

import numpy as np
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.signal import savgol_filter
import os, warnings

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
TICKERS = [
    # US Large Cap Tech
    "AAPL", "MSFT", "GOOG", "AMZN", "NVDA",
    "META", "TSLA", "NFLX", "AMD", "INTC",

    # US Financials
    "JPM", "BAC", "GS", "MS", "WFC",
    "BLK", "C", "AXP", "V", "MA",

    # US Healthcare & Consumer
    "JNJ", "PFE", "UNH", "MRK", "ABBV",
    "PG", "KO", "PEP", "WMT", "COST",

    # US Energy & Industrials
    "XOM", "CVX", "COP", "BA", "CAT",
    "GE", "HON", "LMT", "RTX", "UPS",

    # Commodities & Crypto (yfinance format)
    "GC=F",    # Gold
    "SI=F",    # Silver
    "CL=F",    # Crude Oil
    "BTC-USD", # Bitcoin
    "ETH-USD", # Ethereum

    # International ETFs (broad exposure)
    "EWJ",  # Japan
    "EWZ",  # Brazil
    "FXI",  # China
    "EWG",  # Germany
    "EWU",  # UK
]
START     = "2010-01-01"
END       = "2026-04-11"

Q_VALS    = np.arange(-30, 31, 5, dtype=float)
SCALE_MIN = 20
N_SCALES  = 30
POLY_ORD  = 1
SCALE_MAX = 800
OVERLAP   = 0.75

OUT_DIR   = "mf_plots"
os.makedirs(OUT_DIR, exist_ok=True)

PAPER_COLORS = [
    "#d62728", "#2ca02c", "#1f77b4", "#000000", "#9467bd",
    "#ff7f0e", "#8c564b", "#e377c2", "#17becf", "#bcbd22", "#7f7f7f",
]

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def prepare_series(price):
    price     = price[price > 0]
    log_price = np.log(price)
    returns   = np.diff(log_price)
    return price, log_price, returns


def integrate(x):
    """
    DFA profile: Y(i) = sum_{k=1}^{i} [x(k) - mean(x)]
    Converts stationary returns into a random-walk-like profile
    with genuine scale-dependent variance.
    Used only by MF-DFA.
    """
    return np.cumsum(x - x.mean())


def _overlap_starts(N, s, overlap):
    step = max(1, int(s * (1.0 - overlap)))
    return np.arange(0, N - s + 1, step)


# ─────────────────────────────────────────────────────────────────────────────
# MF-DFA  — input: log-returns
# ─────────────────────────────────────────────────────────────────────────────
def mfdfa(x, q_vals, scale_min, scale_max, n_scales, poly_ord=1, overlap=0.75):
    """
    MF-DFA on log-returns:
      1. Integrate returns → profile Y
      2. For each scale s, detrend each overlapping segment with a polynomial
      3. RMS of residuals → local fluctuation F_v(s)
      4. q-th order average over segments → F_q(s)
      5. Slope of log F_q vs log s → H(q)
    """
    y = integrate(x)
    N = len(y)
    scale_max = min(scale_max, N // 4)
    scale_min = max(scale_min, poly_ord + 2)

    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales).astype(int)
    )
    scales = scales[scales >= scale_min]

    Fqs = np.full((len(q_vals), len(scales)), np.nan)

    for i, s in enumerate(scales):
        starts = _overlap_starts(N, s, overlap)
        if len(starts) < 4:
            continue

        t   = np.arange(s)
        rms = []
        for st in starts:
            seg  = y[st : st + s]
            coef = np.polyfit(t, seg, poly_ord)
            r    = np.sqrt(np.mean((seg - np.polyval(coef, t)) ** 2))
            if r > 0:
                rms.append(r)

        rms = np.asarray(rms)
        if len(rms) == 0:
            continue

        for j, q in enumerate(q_vals):
            if q == 0:
                Fqs[j, i] = np.exp(0.5 * np.mean(np.log(rms ** 2)))
            else:
                Fqs[j, i] = (np.mean(rms ** q)) ** (1.0 / q)

    hq = np.full(len(q_vals), np.nan)
    for j in range(len(q_vals)):
        row  = Fqs[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() >= 4:
            hq[j] = np.polyfit(np.log(scales[mask]), np.log(row[mask]), 1)[0]

    return scales, Fqs, hq


# ─────────────────────────────────────────────────────────────────────────────
# MF-DMA  — input: log-prices
# ─────────────────────────────────────────────────────────────────────────────
def mfdma(x, q_vals, scale_min, scale_max, n_scales, theta=0.5, overlap=0.75):
    """
    MF-DMA on log-prices:
      NO integration step — log-prices already carry the slow trend
      that the moving average will track and subtract.

      1. For each scale s, compute moving average of log-prices
         as the local trend (the MA naturally follows the price trend)
      2. Residual = log-price - moving average = detrended fluctuation
      3. q-th order average of squared residuals → F_q(s)
      4. Slope of log F_q vs log s → H(q)

    theta controls moving average asymmetry:
      0.0 → backward MA  (uses only past values, causal)
      0.5 → centred MA   (symmetric, non-causal)
      1.0 → forward MA   (uses only future values)

    H(q) here will be > 1 due to log-price nonstationarity —
    this is expected and consistent with DMA literature.
    The key output is the SHAPE of H(q) and Δα, not the absolute level.
    """
    N = len(x)
    scale_max = min(scale_max, N // 4)
    scale_min = max(scale_min, 4)

    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales).astype(int)
    )
    scales = scales[scales >= scale_min]

    Fqs = np.full((len(q_vals), len(scales)), np.nan)

    for i, s in enumerate(scales):
        starts = _overlap_starts(N, s, overlap)
        if len(starts) < 4:
            continue

        k_back = int(np.floor((s - 1) * theta))
        k_fwd  = int(np.ceil ((s - 1) * (1.0 - theta)))

        sq_res_all = []
        for st in starts:
            seg_end = st + s
            for k in range(st, seg_end):
                # Moving average window around k, clipped to segment bounds
                w_start = max(st,      k - k_back)
                w_end   = min(seg_end, k + k_fwd + 1)
                window  = x[w_start:w_end]
                if len(window) == 0:
                    continue
                sq_res_all.append((x[k] - window.mean()) ** 2)

        sq_res = np.asarray(sq_res_all)
        sq_res = sq_res[sq_res > 0]
        if len(sq_res) == 0:
            continue

        for j, q in enumerate(q_vals):
            if q == 0:
                Fqs[j, i] = np.exp(0.5 * np.mean(np.log(sq_res)))
            else:
                Fqs[j, i] = (np.mean(sq_res ** (q / 2.0))) ** (1.0 / q)

    hq = np.full(len(q_vals), np.nan)
    for j in range(len(q_vals)):
        row  = Fqs[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() >= 4:
            hq[j] = np.polyfit(np.log(scales[mask]), np.log(row[mask]), 1)[0]

    return scales, Fqs, hq


# ─────────────────────────────────────────────────────────────────────────────
# MULTIFRACTAL SPECTRUM
# ─────────────────────────────────────────────────────────────────────────────
def multifractal_spectrum(q_vals, hq):
    tau_q = q_vals * hq - 1.0
    wl = min(5, len(tau_q) if len(tau_q) % 2 == 1 else len(tau_q) - 1)
    tau_smooth = savgol_filter(tau_q, window_length=wl, polyorder=2) if len(tau_q) >= 5 else tau_q
    alpha   = np.gradient(tau_smooth, q_vals)
    D_alpha = q_vals * alpha - tau_smooth
    return tau_q, alpha, D_alpha


# ─────────────────────────────────────────────────────────────────────────────
# 6-PANEL PLOT
# ─────────────────────────────────────────────────────────────────────────────
SERIES_LABELS = {
    "log_prices": "Log-Prices",
    "returns":    "Log-Returns",
}

def plot_6panel(ticker, method, series_key, raw_series,
                scales, Fqs, hq, q_vals, tau_q, alpha, D_alpha):

    series_label = SERIES_LABELS[series_key]
    valid  = np.isfinite(hq)
    colors = (PAPER_COLORS * 4)[: len(q_vals)]

    fig, axs = plt.subplots(2, 3, figsize=(15, 9))
    '''fig.suptitle(
        f"Multifractal analysis  ·  {ticker}  ·  {series_label}  ·  {method}",
        fontsize=13, fontweight="bold", y=1.01,
    )'''

    # (a) raw series
    ax = axs[0, 0]
    ax.plot(raw_series, color="#c0392b", linewidth=0.65)
    ax.set_xlabel("$t$", fontsize=11)
    ax.set_ylabel("$X(t)$", fontsize=11)
    ax.set_title("(a) Time series", fontsize=11)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.4g"))

    # (b) F_q(s) log-log
    ax = axs[0, 1]
    legend_handles = []
    for idx_q, (q, c) in enumerate(zip(q_vals, colors)):
        row  = Fqs[idx_q]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() < 2:
            continue
        h, = ax.loglog(scales[mask], row[mask], "o-",
                       color=c, markersize=3.5, linewidth=1.0)
        legend_handles.append((h, f"$q={int(q)}$"))

    ax.set_xlabel("$s$", fontsize=11)
    ax.set_ylabel("$F_q(s)$", fontsize=11)
    ax.set_title("(b) Fluctuation functions $F_q(s)$", fontsize=11)
    step  = max(1, len(legend_handles) // 8)
    shown = legend_handles[::step]
    ax.legend([h for h, _ in shown], [l for _, l in shown],
              fontsize=7, ncol=1, loc="best",
              handlelength=1.2, handletextpad=0.4)

    # (c) H(q)
    ax = axs[0, 2]
    ax.plot(q_vals[valid], hq[valid], "-", color="#1f4e8c", linewidth=2)
    ax.set_xlabel("$q$", fontsize=11)
    ax.set_ylabel("$H(q)$", fontsize=11)
    ax.set_title("(c) Generalised Hurst exponent $H(q)$", fontsize=11)
    ax.axhline(0.5, color="grey", lw=0.8, ls="--", alpha=0.6, label="H=0.5")
    ax.axhline(1.0, color="blue", lw=0.8, ls=":",  alpha=0.6, label="H=1.0")
    ax.legend(fontsize=7)

    # (d) tau(q)
    ax = axs[1, 0]
    ax.plot(q_vals[valid], tau_q[valid], "-", color="#1f4e8c", linewidth=2)
    ax.set_xlabel("$q$", fontsize=11)
    ax.set_ylabel(r"$\tau(q)$", fontsize=11)
    ax.set_title(r"(d) Scaling exponent $\tau(q)$", fontsize=11)
    ax.axhline(0, color="grey", lw=0.8, ls="--", alpha=0.6)

    # (e) alpha(q)
    ax = axs[1, 1]
    valid_s = np.isfinite(alpha) & np.isfinite(D_alpha)
    ax.plot(q_vals[valid_s], alpha[valid_s], "-", color="#1f4e8c", linewidth=2)
    ax.set_xlabel("$q$", fontsize=11)
    ax.set_ylabel(r"$\alpha(q)$", fontsize=11)
    ax.set_title(r"(e) Singularity strength $\alpha(q)$", fontsize=11)

    # (f) f(alpha)
    ax = axs[1, 2]
    ax.plot(alpha[valid_s], D_alpha[valid_s], "-", color="#1f4e8c", linewidth=2)
    q0_idx = np.argmin(np.abs(q_vals))
    if np.isfinite(alpha[q0_idx]) and np.isfinite(D_alpha[q0_idx]):
        ax.plot(alpha[q0_idx], D_alpha[q0_idx], "o",
                color="#c0392b", markersize=6, zorder=5)
    ax.set_xlabel(r"$\alpha$", fontsize=11)
    ax.set_ylabel(r"$f(\alpha)$", fontsize=11)
    ax.set_title(r"(f) Multifractal spectrum $f(\alpha)$", fontsize=11)

    for ax in axs.ravel():
        ax.tick_params(direction="in", which="both", top=True, right=True)
        ax.spines[["top", "right"]].set_visible(True)

    if valid_s.sum() >= 2:
        delta_alpha = alpha[valid_s].max() - alpha[valid_s].min()
        fig.text(0.5, -0.01,
                 fr"Multifractal width  $\Delta\alpha = {delta_alpha:.4f}$",
                 ha="center", fontsize=10, color="grey")
    print(f"{ticker}'s Multifractal width: {delta_alpha:.4f}")
    plt.tight_layout()
    fname = (f"{OUT_DIR}/{ticker.replace('=', '_')}"
             f"_{method.replace('-', '_')}_{series_key}.png")
    plt.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"    saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
# Each method gets the theoretically correct input series:
#   MF-DFA  → log-returns  (stationary, polynomial detrending after integration)
#   DMA variants → log-prices  (nonstationary, MA naturally tracks the trend)

DMA_VARIANTS = {
    "MF-DFA":  (None, "returns"),     # (theta, series_key)
    #"MF-BDMA": (0.0,  "log_prices"),
    #"MF-CDMA": (0.5,  "log_prices"),
    #"MF-FDMA": (1.0,  "log_prices"),
}

for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"Ticker: {ticker}")

    data  = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=True)
    close = data["Close"].dropna().values.ravel().astype(float)

    if len(close) < 200:
        print("  Not enough data, skipping.")
        continue

    _, log_price, returns = prepare_series(close)

    series_map = {
        "returns":    returns,
        "log_prices": log_price,
    }

    for method, (theta, series_key) in DMA_VARIANTS.items():
        x = series_map[series_key]
        N = len(x)
        scale_max_eff = min(SCALE_MAX, N // 4)
        scale_min_eff = max(SCALE_MIN, POLY_ORD + 2)

        print(f"  {method} on {series_key}  (N={N}, "
              f"scales [{scale_min_eff}, {scale_max_eff}])")

        if method == "MF-DFA":
            scales, Fqs, hq = mfdfa(
                x, Q_VALS,
                scale_min_eff, scale_max_eff, N_SCALES,
                poly_ord=POLY_ORD, overlap=OVERLAP,
            )
        else:
            scales, Fqs, hq = mfdma(
                x, Q_VALS,
                scale_min_eff, scale_max_eff, N_SCALES,
                theta=theta, overlap=OVERLAP,
            )

        tau_q, alpha, D_alpha = multifractal_spectrum(Q_VALS, hq)

        plot_6panel(
            ticker, method, series_key, x,
            scales, Fqs, hq, Q_VALS,
            tau_q, alpha, D_alpha,
        )

print("\nAll done. Figures saved in:", OUT_DIR)


Ticker: AAPL
  MF-DFA on returns  (N=4091, scales [20, 800])
AAPL's Multifractal width: 0.4912
    saved → mf_plots/AAPL_MF_DFA_returns.png

Ticker: MSFT
  MF-DFA on returns  (N=4091, scales [20, 800])
MSFT's Multifractal width: 0.4881
    saved → mf_plots/MSFT_MF_DFA_returns.png

Ticker: GOOG
  MF-DFA on returns  (N=4091, scales [20, 800])
GOOG's Multifractal width: 0.3538
    saved → mf_plots/GOOG_MF_DFA_returns.png

Ticker: AMZN
  MF-DFA on returns  (N=4091, scales [20, 800])
AMZN's Multifractal width: 0.4870
    saved → mf_plots/AMZN_MF_DFA_returns.png

Ticker: NVDA
  MF-DFA on returns  (N=4091, scales [20, 800])
NVDA's Multifractal width: 0.1812
    saved → mf_plots/NVDA_MF_DFA_returns.png

Ticker: META
  MF-DFA on returns  (N=3492, scales [20, 800])
META's Multifractal width: 0.4263
    saved → mf_plots/META_MF_DFA_returns.png

Ticker: TSLA
  MF-DFA on returns  (N=3969, scales [20, 800])
TSLA's Multifractal width: 0.5141
    saved → mf_plots/TSLA_MF_DFA_returns.png

Ticker: NFLX

In [9]:
"""
Sensitivity sweep for the Rosenstein LLE chaos test across (m, tau) --
v2: decoupled Theiler window + return/squared-return ACF diagnostic.

CHANGES FROM v1:
  1. Theiler window is now FIXED (THEILER_FIXED, default 20) and no longer
     tied to tau via max(tau, 5). This isolates whether the tau-dependence
     seen in v1 (0 BH hits at tau<=5, 11 at tau=7, 33 at tau=10, all at m=10)
     is a genuine embedding-delay effect or an artifact of the exclusion
     window growing alongside tau.
  2. After the sweep, computes the autocorrelation function (ACF) of raw
     returns AND squared returns out to LAG_MAX (default 15) for a fixed
     set of tickers (the 8 most BH-robust tickers from v1: AAPL, JNJ, EWJ,
     V, PG, PFE, BTC-USD, COST), to check independently whether there is a
     visible bump in serial dependence around lag 7-10 that would explain
     why LLE significance in v1 turned on specifically at tau>=7.

Everything else (data caching, GPU Rosenstein/IAAFT pipeline, BH correction,
9-point sweep grid) is unchanged from v1 so this is a single-variable
comparison.
"""

import time
import numpy as np
import pandas as pd
import torch
import yfinance as yf
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

DTYPE = torch.float32

START_DATE = "2010-01-01"
END_DATE = "2026-04-11"
DATA_DIR = Path("stock_csv_data")
DATA_DIR.mkdir(exist_ok=True)

TICKERS = [
    "AAPL","MSFT","GOOG","AMZN","NVDA","META","TSLA","NFLX","AMD","INTC",
    "JPM","BAC","GS","MS","WFC","BLK","C","AXP","V","MA",
    "JNJ","PFE","UNH","MRK","ABBV","PG","KO","PEP","WMT","COST",
    "XOM","CVX","COP","BA","CAT","GE","HON","LMT","RTX","UPS",
    "GC=F","SI=F","CL=F","BTC-USD","ETH-USD",
    "EWJ","EWZ","FXI","EWG","EWU",
]

N_SURROGATES = 200
SEED = 42
SURROGATE_CHUNK = 16   # lower if you OOM, raise if you have GPU headroom

# ── CHANGE 1: fixed Theiler window, decoupled from tau ─────────────────
THEILER_FIXED = 20     # was: theiler = max(tau, 5)

# baseline + one-parameter-at-a-time sweep, same style as Table sci_sensitivity
BASELINE = (7, 7)
M_SWEEP = [(m, 10) for m in [4, 6, 8, 10, 12]]
TAU_SWEEP = [(10, t) for t in [1, 3, 5, 7, 10]]
PARAM_GRID = sorted(set(M_SWEEP + TAU_SWEEP))  # dedupe shared baseline

# ── CHANGE 2: ACF diagnostic config ─────────────────────────────────────
ACF_TICKERS = ["AAPL", "JNJ", "EWJ", "V", "PG", "PFE", "BTC-USD", "COST"]
LAG_MAX = 15


# ─────────────────────────────────────────────────────────────────────
# Data (cached -- reuses your existing stock_csv_data/ CSVs)
# ─────────────────────────────────────────────────────────────────────

def get_returns(ticker: str) -> np.ndarray:
    csv_path = DATA_DIR / f"{ticker.replace('=', '_')}_data.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        close = pd.to_numeric(df["Close"], errors="coerce").dropna()
    else:
        print(f"  downloading {ticker} ...")
        df = yf.download(ticker, start=START_DATE, end=END_DATE,
                          auto_adjust=True, progress=False)
        if df.empty:
            return np.array([])
        df = df.reset_index()
        df.to_csv(csv_path, index=False)
        close = pd.to_numeric(df["Close"], errors="coerce").dropna()
    close = close.values.astype(np.float64)
    if len(close) < 500:
        return np.array([])
    return np.diff(np.log(close))


# ─────────────────────────────────────────────────────────────────────
# Core GPU-vectorized Rosenstein LLE + IAAFT surrogates
# (identical numerics to the validated lyapunov_test_gpu.py, except the
#  Theiler window is now a fixed constant passed in, not derived from tau)
# ─────────────────────────────────────────────────────────────────────

def embed_batch(X: torch.Tensor, m: int, tau: int) -> torch.Tensor:
    B, N = X.shape
    n_vec = N - (m - 1) * tau
    idx = torch.arange(n_vec, device=X.device)[:, None] + tau * torch.arange(m, device=X.device)[None, :]
    return X[:, idx]


def rosenstein_lle_batch(X: torch.Tensor, m: int, tau: int, theiler: int,
                          max_iter: int) -> torch.Tensor:
    B, N = X.shape
    emb = embed_batch(X, m, tau)
    n_vec = emb.shape[1]

    dist = torch.cdist(emb, emb)
    ar = torch.arange(n_vec, device=X.device)
    theiler_mask = (ar[:, None] - ar[None, :]).abs() <= theiler
    dist = dist.masked_fill(theiler_mask[None, :, :], float("inf"))

    nn_dist, nn_idx = dist.min(dim=2)
    valid_start = torch.isfinite(nn_dist)

    max_iter = min(max_iter, n_vec - 1)
    log_div_sum = torch.zeros(B, max_iter, device=X.device)
    counts = torch.zeros(B, max_iter, device=X.device)

    i_idx = ar[None, :].expand(B, n_vec)
    j_idx = nn_idx

    for k in range(max_iter):
        ii = i_idx + k
        jj = j_idx + k
        ok = (ii < n_vec) & (jj < n_vec) & valid_start
        if ok.sum() == 0:
            break
        ii_c = torch.where(ok, ii, torch.zeros_like(ii))
        jj_c = torch.where(ok, jj, torch.zeros_like(jj))
        pts_i = torch.gather(emb, 1, ii_c[..., None].expand(-1, -1, m))
        pts_j = torch.gather(emb, 1, jj_c[..., None].expand(-1, -1, m))
        d = torch.linalg.vector_norm(pts_i - pts_j, dim=-1)
        d = torch.where(ok & (d > 1e-10), d, torch.full_like(d, float("nan")))
        log_d = torch.log(d)
        valid_k = torch.isfinite(log_d)
        log_div_sum[:, k] = torch.nan_to_num(log_d, nan=0.0).sum(dim=1)
        counts[:, k] = valid_k.sum(dim=1).float()

    min_count = n_vec * 0.05
    valid_k_mask = counts > min_count
    mean_log_div = torch.where(valid_k_mask, log_div_sum / counts.clamp(min=1),
                                torch.full_like(counts, float("nan")))

    lambdas = torch.full((B,), float("nan"), device=X.device)
    ks = torch.arange(max_iter, device=X.device).float()
    for b in range(B):
        vk = valid_k_mask[b]
        if vk.sum() < 5:
            continue
        y = mean_log_div[b][vk]
        x_ = ks[vk]
        cutoff = max(5, int(0.6 * len(x_)))
        x_fit, y_fit = x_[:cutoff], y[:cutoff]
        x_mean, y_mean = x_fit.mean(), y_fit.mean()
        num = ((x_fit - x_mean) * (y_fit - y_mean)).sum()
        den = ((x_fit - x_mean) ** 2).sum()
        if den > 1e-12:
            lambdas[b] = num / den
    return lambdas


def iaaft_surrogates_batch(x: np.ndarray, n_surrogates: int, n_iter: int = 100,
                            seed: int = 42) -> torch.Tensor:
    rng = np.random.default_rng(seed)
    N = len(x)
    x_t = torch.tensor(x, dtype=DTYPE, device=DEVICE)
    sorted_x = torch.sort(x_t)[0]
    target_amp = torch.abs(torch.fft.fft(x_t))

    init = np.stack([rng.permutation(x) for _ in range(n_surrogates)])
    surrogate = torch.tensor(init, dtype=DTYPE, device=DEVICE)
    target_amp_b = target_amp[None, :].expand(n_surrogates, -1)

    for _ in range(n_iter):
        s_fft = torch.fft.fft(surrogate)
        phases = torch.angle(s_fft)
        s_fft_new = target_amp_b * torch.exp(1j * phases)
        s_new = torch.real(torch.fft.ifft(s_fft_new))
        ranks = torch.argsort(torch.argsort(s_new, dim=1), dim=1)
        surrogate = sorted_x[ranks]

    return surrogate


def test_ticker_chaos(returns: np.ndarray, ticker: str, m: int, tau: int,
                       n_surrogates: int = N_SURROGATES, seed: int = SEED):
    x = returns[np.isfinite(returns)]
    if len(x) < 500:
        return {"ticker": ticker, "m": m, "tau": tau, "status": "insufficient_data"}

    # ── CHANGE 1 applied here: fixed Theiler window, not max(tau, 5) ──
    theiler = THEILER_FIXED
    max_iter = min(theiler * 10, (len(x) - (m - 1) * tau) // 4)
    if max_iter < 5:
        return {"ticker": ticker, "m": m, "tau": tau, "status": "series_too_short_for_params"}

    x_real = torch.tensor(x, dtype=DTYPE, device=DEVICE)[None, :]
    lam_obs = rosenstein_lle_batch(x_real, m, tau, theiler, max_iter).item()
    if not np.isfinite(lam_obs):
        return {"ticker": ticker, "m": m, "tau": tau, "status": "estimation_failed"}

    surrogates = iaaft_surrogates_batch(x, n_surrogates, n_iter=100, seed=seed)
    lam_null_chunks = []
    for start in range(0, n_surrogates, SURROGATE_CHUNK):
        chunk = surrogates[start:start + SURROGATE_CHUNK]
        lam_null_chunks.append(rosenstein_lle_batch(chunk, m, tau, theiler, max_iter))
    lam_null = torch.cat(lam_null_chunks)
    lam_null = lam_null[torch.isfinite(lam_null)].cpu().numpy()

    if len(lam_null) < 20:
        return {"ticker": ticker, "m": m, "tau": tau, "status": "surrogate_unstable",
                "lambda_obs": lam_obs}

    p_val = (np.sum(lam_null >= lam_obs) + 1) / (len(lam_null) + 1)
    z = (lam_obs - lam_null.mean()) / (lam_null.std() + 1e-12)

    return {
        "ticker": ticker, "m": m, "tau": tau, "status": "ok",
        "lambda_obs": lam_obs, "surrogate_mean": lam_null.mean(),
        "surrogate_std": lam_null.std(), "z_score": z, "p_value": p_val,
        "n_surrogates_valid": len(lam_null),
    }


# ─────────────────────────────────────────────────────────────────────
# Benjamini-Hochberg (same convention as the paper's Delta_alpha_corr /
# SCI tables)
# ─────────────────────────────────────────────────────────────────────

def bh_correct(pvals: np.ndarray, alpha: float = 0.05):
    n = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    thresh = (np.arange(1, n + 1) / n) * alpha
    passing = ranked <= thresh
    if not passing.any():
        return np.full(n, False), np.minimum.accumulate((ranked * n / np.arange(1, n + 1))[::-1])[::-1][np.argsort(order)]
    max_pass_rank = np.max(np.where(passing)[0])
    reject = np.zeros(n, dtype=bool)
    reject[order[: max_pass_rank + 1]] = True
    # BH-adjusted q-values (monotone)
    q_sorted = np.minimum.accumulate((ranked * n / np.arange(1, n + 1))[::-1])[::-1]
    q_sorted = np.clip(q_sorted, 0, 1)
    q = np.empty(n)
    q[order] = q_sorted
    return reject, q


# ─────────────────────────────────────────────────────────────────────
# CHANGE 2: ACF / squared-return ACF diagnostic
# ─────────────────────────────────────────────────────────────────────

def acf(x: np.ndarray, lag_max: int) -> np.ndarray:
    """Sample autocorrelation function, lags 1..lag_max, via numpy only
    (no external stats dependency needed)."""
    x = x - x.mean()
    n = len(x)
    denom = np.sum(x ** 2)
    out = np.empty(lag_max)
    for lag in range(1, lag_max + 1):
        out[lag - 1] = np.sum(x[:-lag] * x[lag:]) / denom
    return out


def run_acf_diagnostic(returns_cache: dict, tickers: list, lag_max: int) -> pd.DataFrame:
    """
    For each ticker, computes the ACF of raw returns (linear serial
    dependence) and of squared returns (volatility clustering / nonlinear
    dependence) out to lag_max. Purpose: check independently whether there
    is a visible bump in serial dependence around lag 7-10 -- the same
    region where the v1 LLE sweep's BH significance turned on -- which
    would support reading the LLE result (if it survives the fixed-Theiler
    rerun) as detecting known correlation structure rather than a novel
    chaotic signature.
    """
    rows = []
    for ticker in tickers:
        if ticker not in returns_cache:
            print(f"  [ACF] skipping {ticker}: no cached returns")
            continue
        x = returns_cache[ticker]
        x = x[np.isfinite(x)]
        raw_acf = acf(x, lag_max)
        sq_acf = acf(x ** 2, lag_max)
        # approximate 95% white-noise confidence band: +/- 1.96/sqrt(N)
        band = 1.96 / np.sqrt(len(x))
        for lag in range(1, lag_max + 1):
            rows.append({
                "ticker": ticker,
                "lag": lag,
                "acf_return": raw_acf[lag - 1],
                "acf_sq_return": sq_acf[lag - 1],
                "sig_return": abs(raw_acf[lag - 1]) > band,
                "sig_sq_return": abs(sq_acf[lag - 1]) > band,
                "ci_band_95": band,
            })
    return pd.DataFrame(rows)


# ─────────────────────────────────────────────────────────────────────
# Main sweep
# ─────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    t_start = time.time()
    print(f"Loading/caching returns for {len(TICKERS)} tickers ...")
    returns_cache = {}
    for ticker in TICKERS:
        r = get_returns(ticker)
        if len(r) > 0:
            returns_cache[ticker] = r
    print(f"  {len(returns_cache)} tickers with usable data.\n")

    print(f"Theiler window fixed at {THEILER_FIXED} (decoupled from tau) -- v2 change.\n")
    print(f"Sweeping {len(PARAM_GRID)} (m, tau) specifications: {PARAM_GRID}\n")

    all_rows = []
    spec_summaries = []

    for spec_i, (m, tau) in enumerate(PARAM_GRID):
        print(f"=== Specification {spec_i+1}/{len(PARAM_GRID)}: m={m}, tau={tau} "
              f"(theiler={THEILER_FIXED}) ===")
        t0 = time.time()
        spec_rows = []
        for ticker, returns in returns_cache.items():
            res = test_ticker_chaos(returns, ticker, m, tau)
            spec_rows.append(res)
        elapsed = time.time() - t0

        spec_df = pd.DataFrame(spec_rows)
        ok = spec_df[spec_df["status"] == "ok"].copy()
        if len(ok) > 0:
            reject, q = bh_correct(ok["p_value"].values)
            ok["bh_reject"] = reject
            ok["q_value"] = q
        else:
            ok["bh_reject"] = []
            ok["q_value"] = []

        n_nominal = (ok["p_value"] < 0.05).sum() if len(ok) else 0
        n_bh = ok["bh_reject"].sum() if len(ok) else 0
        bh_tickers = ok.loc[ok["bh_reject"], "ticker"].tolist() if len(ok) else []

        print(f"  {len(ok)} tickers estimated OK in {elapsed:.1f}s | "
              f"{n_nominal} nominal (p<0.05) | {n_bh} survive BH correction"
              + (f" -> {bh_tickers}" if bh_tickers else ""))
        print()

        all_rows.append(ok)
        spec_summaries.append({
            "m": m, "tau": tau, "theiler": THEILER_FIXED, "n_tickers_ok": len(ok),
            "n_nominal_p05": n_nominal, "n_bh_significant": n_bh,
            "bh_significant_tickers": ";".join(bh_tickers),
        })

    full_df = pd.concat(all_rows, ignore_index=True)
    full_df.to_csv("lyapunov_sensitivity_full_v2.csv", index=False)

    spec_df_summary = pd.DataFrame(spec_summaries)
    spec_df_summary.to_csv("lyapunov_sensitivity_spec_summary_v2.csv", index=False)

    # per-ticker robustness score: in how many of the 9 specs was this
    # ticker nominally significant / BH-significant?
    n_specs = len(PARAM_GRID)
    ticker_scores = (
        full_df.assign(nominal=lambda d: d["p_value"] < 0.05)
        .groupby("ticker")
        .agg(n_specs_present=("m", "count"),
             n_nominal_hits=("nominal", "sum"),
             n_bh_hits=("bh_reject", "sum"))
        .reset_index()
        .sort_values("n_bh_hits", ascending=False)
    )
    ticker_scores.to_csv("lyapunov_sensitivity_ticker_summary_v2.csv", index=False)

    print("=" * 70)
    print("SPECIFICATION SUMMARY (v2, fixed Theiler window = "
          f"{THEILER_FIXED}, BH-corrected within spec)")
    print(spec_df_summary.to_string(index=False))
    print()
    print("PER-TICKER ROBUSTNESS (how many of the", n_specs,
          "specifications flagged each ticker)")
    print(ticker_scores[ticker_scores["n_nominal_hits"] > 0].to_string(index=False))
    print()

    any_bh_ever = ticker_scores["n_bh_hits"].sum()
    max_bh_any_ticker = ticker_scores["n_bh_hits"].max() if len(ticker_scores) else 0
    print(f"Total LLE-sweep wall time: {time.time() - t_start:.1f}s")
    if any_bh_ever == 0:
        print("No ticker survives Benjamini-Hochberg correction under ANY of the "
              f"{n_specs} (m, tau) specifications tested, with the Theiler window "
              "now fixed and decoupled from tau. Combined with the v1 result, this "
              "indicates the earlier tau-dependent significance was an artifact of "
              "coupling the Theiler exclusion window to tau, not a genuine "
              "embedding-delay effect: no robust evidence of chaos (in the "
              "Rosenstein-LLE sense) is detected for this asset universe.")
    elif max_bh_any_ticker >= n_specs * 0.6:
        print("At least one ticker survives BH correction in a majority of "
              "specifications even with a fixed Theiler window -- the "
              "tau-dependence pattern from v1 appears to be a genuine effect, "
              "not an artifact of the theiler-tau coupling. Worth comparing "
              "spec-by-spec against lyapunov_sensitivity_spec_summary.csv (v1) "
              "to see whether the SAME (m, tau) specs turn significant.")
    else:
        print("Some BH-significant hits appear, but not consistently across "
              "specifications -- compare directly against the v1 spec summary "
              "to determine whether the tau-dependence pattern weakened once "
              "the Theiler window was decoupled from tau.")

    # ── CHANGE 2: run and report the ACF diagnostic ─────────────────────
    print()
    print("=" * 70)
    print(f"ACF / SQUARED-RETURN ACF DIAGNOSTIC (lags 1-{LAG_MAX})")
    print(f"Tickers: {ACF_TICKERS}")
    print("Purpose: check independently whether serial dependence (linear or "
          "volatility-clustering) shows a visible bump around lag 7-10, the "
          "region where v1's LLE significance turned on.\n")

    acf_df = run_acf_diagnostic(returns_cache, ACF_TICKERS, LAG_MAX)
    acf_df.to_csv("acf_diagnostic.csv", index=False)

    # print a compact wide table: squared-return ACF (volatility clustering)
    # is usually the more informative one for this question
    pivot_sq = acf_df.pivot(index="lag", columns="ticker", values="acf_sq_return")
    pivot_sq = pivot_sq[[t for t in ACF_TICKERS if t in pivot_sq.columns]]
    print("Squared-return ACF by lag (volatility-clustering signal):")
    print(pivot_sq.round(4).to_string())
    print()

    pivot_raw = acf_df.pivot(index="lag", columns="ticker", values="acf_return")
    pivot_raw = pivot_raw[[t for t in ACF_TICKERS if t in pivot_raw.columns]]
    print("Raw-return ACF by lag (linear serial dependence):")
    print(pivot_raw.round(4).to_string())
    print()

    # flag any ticker with a local bump (higher |acf| at lag 7-10 than at
    # lags 1-3) in either series, as a quick automated signpost
    print("Bump check (mean |ACF| at lags 7-10 vs mean |ACF| at lags 1-3):")
    for ticker in ACF_TICKERS:
        if ticker not in pivot_sq.columns:
            continue
        sq_early = pivot_sq[ticker].loc[1:3].abs().mean()
        sq_late = pivot_sq[ticker].loc[7:10].abs().mean()
        raw_early = pivot_raw[ticker].loc[1:3].abs().mean()
        raw_late = pivot_raw[ticker].loc[7:10].abs().mean()
        flag_sq = " <-- BUMP" if sq_late > sq_early else ""
        flag_raw = " <-- BUMP" if raw_late > raw_early else ""
        print(f"  {ticker:8s} sq-return: lags1-3={sq_early:.4f} lags7-10={sq_late:.4f}{flag_sq}"
              f"   |   raw-return: lags1-3={raw_early:.4f} lags7-10={raw_late:.4f}{flag_raw}")

    print()
    print(f"Total wall time (sweep + ACF): {time.time() - t_start:.1f}s")

Using device: cuda
  GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Loading/caching returns for 50 tickers ...
  50 tickers with usable data.

Theiler window fixed at 20 (decoupled from tau) -- v2 change.

Sweeping 9 (m, tau) specifications: [(4, 10), (6, 10), (8, 10), (10, 1), (10, 3), (10, 5), (10, 7), (10, 10), (12, 10)]

=== Specification 1/9: m=4, tau=10 (theiler=20) ===
  50 tickers estimated OK in 112.2s | 32 nominal (p<0.05) | 30 survive BH correction -> ['MSFT', 'GOOG', 'INTC', 'JPM', 'BAC', 'MS', 'WFC', 'C', 'AXP', 'MA', 'PFE', 'UNH', 'PG', 'KO', 'WMT', 'COST', 'XOM', 'CVX', 'COP', 'BA', 'CAT', 'GE', 'HON', 'RTX', 'SI=F', 'CL=F', 'BTC-USD', 'EWJ', 'EWZ', 'EWG']

=== Specification 2/9: m=6, tau=10 (theiler=20) ===
  50 tickers estimated OK in 111.4s | 45 nominal (p<0.05) | 45 survive BH correction -> ['AAPL', 'MSFT', 'GOOG', 'AMZN', 'NVDA', 'TSLA', 'AMD', 'INTC', 'JPM', 'BAC', 'GS', 'MS', 'WFC', 'BLK', 'C', 'AXP', 'V', 'MA', 'JNJ', 'PFE', 'UNH', 'MRK', 'ABBV', 'PG', 'KO', 'PEP', 'WMT

In [7]:
"""
Surrogate-based significance test for Δα (spectral width) — parallelized safely
=======================================================================
Fix: yfinance is not thread-safe (shared._DFS dict caused RuntimeError
under backend="threading"). Solution: download all data sequentially
first, THEN parallelize only the CPU-bound MF-DFA/surrogate computation
using backend="loky" (separate processes, no shared mutable state).
"""

import os
import time
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import false_discovery_control
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")

N_SURROGATES = 500
N_JOBS       = -1
RNG_SEED     = 42

DATA_DIR = r"D:\Vedant's Stuff\Finance\Decay_Someday\STOCK PARTICLE\scripts\data"
os.makedirs(DATA_DIR, exist_ok=True)


def delta_alpha_from_alpha(alpha, D_alpha):
    valid = np.isfinite(alpha) & np.isfinite(D_alpha)
    if valid.sum() < 2:
        return np.nan
    return alpha[valid].max() - alpha[valid].min()


def shuffle_returns(returns, rng):
    shuffled = returns.copy()
    rng.shuffle(shuffled)
    return shuffled


def surrogate_delta_alphas(returns, q_vals, scale_min, scale_max, n_scales,
                            poly_ord, overlap, n_surrogates, rng):
    widths = np.full(n_surrogates, np.nan)
    for i in range(n_surrogates):
        x_shuf = shuffle_returns(returns, rng)
        _, _, hq_s = mfdfa(x_shuf, q_vals, scale_min, scale_max, n_scales,
                            poly_ord=poly_ord, overlap=overlap)
        _, alpha_s, D_alpha_s = multifractal_spectrum(q_vals, hq_s)
        widths[i] = delta_alpha_from_alpha(alpha_s, D_alpha_s)
    return widths


def process_returns(ticker, returns, seed_offset):
    """Pure-compute step — no I/O, no shared state, safe to run in
    separate processes."""
    rng = np.random.default_rng(RNG_SEED + seed_offset)

    N = len(returns)
    scale_max_eff = min(SCALE_MAX, N // 4)
    scale_min_eff = max(SCALE_MIN, POLY_ORD + 2)

    _, _, hq = mfdfa(returns, Q_VALS, scale_min_eff, scale_max_eff, N_SCALES,
                      poly_ord=POLY_ORD, overlap=OVERLAP)
    _, alpha, D_alpha = multifractal_spectrum(Q_VALS, hq)
    delta_alpha_orig = delta_alpha_from_alpha(alpha, D_alpha)

    shuf_widths = surrogate_delta_alphas(
        returns, Q_VALS, scale_min_eff, scale_max_eff, N_SCALES,
        POLY_ORD, OVERLAP, N_SURROGATES, rng,
    )
    shuf_widths = shuf_widths[np.isfinite(shuf_widths)]

    if len(shuf_widths) == 0 or not np.isfinite(delta_alpha_orig):
        return None

    delta_alpha_shuf_mean = np.mean(shuf_widths)
    delta_alpha_shuf_std  = np.std(shuf_widths, ddof=1)
    delta_alpha_corr      = delta_alpha_orig - delta_alpha_shuf_mean
    corr_share_pct        = 100.0 * delta_alpha_corr / delta_alpha_orig if delta_alpha_orig > 0 else np.nan
    p_value = (np.sum(shuf_widths >= delta_alpha_orig) + 1) / (len(shuf_widths) + 1)

    return dict(
        ticker=ticker,
        delta_alpha=delta_alpha_orig,
        delta_alpha_shuf_mean=delta_alpha_shuf_mean,
        delta_alpha_shuf_std=delta_alpha_shuf_std,
        delta_alpha_corr=delta_alpha_corr,
        corr_share_pct=corr_share_pct,
        p_value=p_value,
        n_surrogates=len(shuf_widths),
    )


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — download all data sequentially (I/O-bound, not parallelized)
# ─────────────────────────────────────────────────────────────────────────────
print("Downloading price data for all tickers...")
returns_by_ticker = {}

for ticker in tqdm(TICKERS):
    data  = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=True)
    close = data["Close"].dropna().values.ravel().astype(float)
    if len(close) < 200:
        print(f"  {ticker}: not enough data, skipping.")
        continue
    _, _, returns = prepare_series(close)
    returns_by_ticker[ticker] = returns

print(f"Downloaded {len(returns_by_ticker)} / {len(TICKERS)} tickers.\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — run MF-DFA + surrogates in parallel (compute-bound, safe to parallelize)
# ─────────────────────────────────────────────────────────────────────────────
print("Running surrogate tests in parallel...")
t0 = time.time()

raw_results = Parallel(n_jobs=N_JOBS, backend="loky")(
    delayed(process_returns)(ticker, returns, i)
    for i, (ticker, returns) in enumerate(returns_by_ticker.items())
)

results = [r for r in raw_results if r is not None]
print(f"\nCompleted in {time.time() - t0:.1f}s using n_jobs={N_JOBS}")

# ─────────────────────────────────────────────────────────────────────────────
# SAVE + BH-FDR CORRECTION
# ─────────────────────────────────────────────────────────────────────────────
df_results = pd.DataFrame(results).sort_values("ticker").reset_index(drop=True)
out_path = f"{DATA_DIR}/delta_alpha_corr_results.csv"

df_results['q_fdr'] = false_discovery_control(df_results['p_value'], method='bh')
df_results.to_csv(out_path, index=False)

print(f"\nAll done. Surrogate results saved → {out_path}")
print(f"\nSignificant at raw p<0.05:        {(df_results['p_value'] < 0.05).sum()} / {len(df_results)} assets")
print(f"Significant at BH-adjusted q<0.05: {(df_results['q_fdr']    < 0.05).sum()} / {len(df_results)} assets")
print(f"Mean correlation-driven share of Δα: {df_results['corr_share_pct'].mean():.1f}%")

  0%|          | 0/50 [00:00<?, ?it/s]

Downloaded 50 / 50 tickers.

Running surrogate tests in parallel...

Completed in 1157.1s using n_jobs=-1

All done. Surrogate results saved → D:\Vedant's Stuff\Finance\Decay_Someday\STOCK PARTICLE\scripts\data/delta_alpha_corr_results.csv

Significant at raw p<0.05:        36 / 50 assets
Significant at BH-adjusted q<0.05: 36 / 50 assets
Mean correlation-driven share of Δα: 26.9%


In [9]:
non_significant = df_results[df_results['q_fdr'] >= 0.05].sort_values('ticker')
print(non_significant[['ticker', 'delta_alpha_corr', 'p_value', 'q_fdr']])

gamma_zero_tickers = {'CAT', 'EWJ', 'GC=F', 'JNJ', 'GOOG', 'PFE', 'SI=F'}
overlap = set(non_significant['ticker']) & gamma_zero_tickers
print(f"\nOverlap with Γ̄=0 tickers: {overlap}")

   ticker  delta_alpha_corr   p_value     q_fdr
2     AMD         -0.004369  0.493014  0.538054
6     BAC          0.047619  0.273453  0.325539
11   CL=F          0.099461  0.097804  0.128690
23   GOOG          0.016501  0.407186  0.473472
26   INTC          0.108302  0.063872  0.086314
28    JPM          0.076952  0.151697  0.189621
32   META         -0.005097  0.522954  0.555140
33    MRK          0.087697  0.115768  0.148421
36   NFLX         -0.184726  0.994012  0.994012
37   NVDA         -0.154736  0.984032  0.994012
42   SI=F          0.010623  0.461078  0.523952
44    UNH          0.001873  0.495010  0.538054
45    UPS          0.062294  0.185629  0.226377
47    WFC         -0.007455  0.532934  0.555140

Overlap with Γ̄=0 tickers: {'SI=F', 'GOOG'}


In [5]:
"""
Pearson correlation between F_{+q}(s) and F_{-q}(s) for q = 1..30
MF-DFA on Log-Returns only | Separate plots per sector

Color scheme: shared positional palette across all sectors.
  Position 0 → green  (AAPL / JPM / JNJ / XOM / GC=F / EWJ)
  Position 1 → blue   (MSFT / BAC / PFE / CVX / SI=F / EWZ)
  Position 2 → red    (GOOG / GS  / UNH / COP / CL=F / FXI)
  ...so the same slot reads the same color in every subplot.
"""

import numpy as np
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os, warnings

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# SHARED POSITIONAL PALETTE  (repeated across every sector)
# ─────────────────────────────────────────────────────────────────────────────
TICKER_PALETTE = [
    "#3fb950",  # 0  green
    "#58a6ff",  # 1  blue
    "#f85149",  # 2  red
    "#e3b341",  # 3  amber
    "#bc8cff",  # 4  purple
    "#ff7f0e",  # 5  orange
    "#17becf",  # 6  teal
    "#f78fb3",  # 7  pink
    "#d4a017",  # 8  gold
    "#a8dadc",  # 9  ice blue
]

# ─────────────────────────────────────────────────────────────────────────────
# SECTORS
# ─────────────────────────────────────────────────────────────────────────────
SECTORS = {
    "Tech": {
        "tickers": ["AAPL", "MSFT", "GOOG", "AMZN", "NVDA",
                    "META", "TSLA", "NFLX", "AMD", "INTC"],
    },
    "Financials": {
        "tickers": ["JPM", "BAC", "GS", "MS", "WFC",
                    "BLK", "C", "AXP", "V", "MA"],
    },
    "Healthcare & Consumer": {
        "tickers": ["JNJ", "PFE", "UNH", "MRK", "ABBV",
                    "PG", "KO", "PEP", "WMT", "COST"],
    },
    "Energy & Industrials": {
        "tickers": ["XOM", "CVX", "COP", "BA", "CAT",
                    "GE", "HON", "LMT", "RTX", "UPS"],
    },
    "Commodities & Crypto": {
        "tickers": ["GC=F", "SI=F", "CL=F", "BTC-USD", "ETH-USD"],
    },
    "International ETFs": {
        "tickers": ["EWJ", "EWZ", "FXI", "EWG", "EWU"],
    },
}

START     = "2010-01-01"
END       = "2026-04-11"
Q_VALS    = np.arange(1, 31, dtype=float)
SCALE_MIN = 20
SCALE_MAX = 800
N_SCALES  = 30
POLY_ORD  = 1
OVERLAP   = 0.75
OUT_DIR   = "mf_corr_plots"
os.makedirs(OUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def prepare_returns(price):
    price = price[price > 0]
    return np.diff(np.log(price))


def integrate(x):
    return np.cumsum(x - x.mean())


def _overlap_starts(N, s, overlap):
    step = max(1, int(s * (1.0 - overlap)))
    return np.arange(0, N - s + 1, step)


def mfdfa_symmetric(x, q_vals, scale_min, scale_max, n_scales,
                    poly_ord=1, overlap=0.75):
    y = integrate(x)
    N = len(y)
    scale_max = min(scale_max, N // 4)
    scale_min = max(scale_min, poly_ord + 2)

    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales).astype(int)
    )
    scales = scales[scales >= scale_min]

    Fqs_pos = np.full((len(q_vals), len(scales)), np.nan)
    Fqs_neg = np.full((len(q_vals), len(scales)), np.nan)

    for i, s in enumerate(scales):
        starts = _overlap_starts(N, s, overlap)
        if len(starts) < 4:
            continue
        t   = np.arange(s)
        rms = []
        for st in starts:
            seg  = y[st: st + s]
            coef = np.polyfit(t, seg, poly_ord)
            r    = np.sqrt(np.mean((seg - np.polyval(coef, t)) ** 2))
            if r > 0:
                rms.append(r)
        rms = np.asarray(rms)
        if len(rms) == 0:
            continue
        for j, q in enumerate(q_vals):
            Fqs_pos[j, i] = (np.mean(rms **  q)) ** (1.0 /  q)
            Fqs_neg[j, i] = (np.mean(rms ** -q)) ** (1.0 / -q)

    return scales, Fqs_pos, Fqs_neg


def pearson_corr(a, b):
    mask = np.isfinite(a) & np.isfinite(b) & (a > 0) & (b > 0)
    if mask.sum() < 3:
        return np.nan
    a_, b_ = np.log(a[mask]), np.log(b[mask])
    cov   = np.mean((a_ - a_.mean()) * (b_ - b_.mean()))
    denom = a_.std() * b_.std()
    if denom == 0:
        return np.nan
    return cov / denom


def _style_ax(ax, sector_name, q_vals, ylim, ytick_step):
    """Apply common axis styling."""
    ax.set_title(sector_name, fontsize=13, fontweight="bold")
    ax.set_xlabel("|q|", fontsize=11)
    ax.set_ylabel("Pearson $r$", fontsize=11)
    ax.set_xticks(q_vals[::2])
    ax.set_xlim(1, 30)
    ax.set_ylim(*ylim)
    ax.yaxis.set_major_locator(plt.MultipleLocator(ytick_step))
    ax.axhline(1.0, color="grey", lw=0.8, ls="--", alpha=0.4)
    ax.axhline(0.0, color="grey", lw=0.8, ls=":",  alpha=0.4)
    ax.tick_params(direction="in", which="both", top=True, right=True)
    ax.legend(fontsize=8.5, ncol=2, loc="lower left", framealpha=0.9)


# ─────────────────────────────────────────────────────────────────────────────
# PRE-COMPUTE all correlation curves (download once, reuse for both figures)
# ─────────────────────────────────────────────────────────────────────────────
# results[sector_name] = list of (ticker, color, corr_vals_array)
results = {}

for sector_name, sector in SECTORS.items():
    tickers = sector["tickers"]
    sector_results = []
    print(f"\n── Sector: {sector_name} ──")

    for t_idx, ticker in enumerate(tickers):
        color = TICKER_PALETTE[t_idx % len(TICKER_PALETTE)]
        print(f"  Processing {ticker} [{color}]...", end=" ")

        data  = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=True)
        close = data["Close"].dropna().values.ravel().astype(float)

        if len(close) < 200:
            print("skipped (< 200 bars).")
            continue

        returns = prepare_returns(close)
        s_min   = max(SCALE_MIN, POLY_ORD + 2)
        s_max   = min(SCALE_MAX, len(returns) // 4)

        scales, Fqs_pos, Fqs_neg = mfdfa_symmetric(
            returns, Q_VALS, s_min, s_max, N_SCALES,
            poly_ord=POLY_ORD, overlap=OVERLAP
        )

        corr_vals = np.array([
            pearson_corr(Fqs_pos[j], Fqs_neg[j])
            for j in range(len(Q_VALS))
        ])

        sector_results.append((ticker, color, corr_vals))
        print(f"min={np.nanmin(corr_vals):.4f}  max={np.nanmax(corr_vals):.4f}")

    results[sector_name] = sector_results


# ─────────────────────────────────────────────────────────────────────────────
# HELPER: build one multi-sector figure
# ─────────────────────────────────────────────────────────────────────────────
def build_figure(ylim, ytick_step):
    n_sectors = len(SECTORS)
    ncols     = 2
    nrows     = int(np.ceil(n_sectors / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows),
                             sharex=True, sharey=True)
    axes = axes.flatten()

    for ax, (sector_name, sector_results) in zip(axes, results.items()):
        for ticker, color, corr_vals in sector_results:
            valid = np.isfinite(corr_vals)
            ax.plot(Q_VALS[valid], corr_vals[valid],
                    "o-", color=color, linewidth=1.8,
                    markersize=4.5, label=ticker)
        _style_ax(ax, sector_name, Q_VALS, ylim, ytick_step)

    for ax in axes[n_sectors:]:
        ax.set_visible(False)

    '''fig.suptitle(
        "Pearson correlation between $F_{+q}(s)$ and $F_{-q}(s)$\n"
        "MF-DFA · Log-Returns · q = 1 to 30  ·  By Sector",
        fontsize=14, fontweight="bold", y=1.01,
    )'''
    
    plt.tight_layout()
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 1 — zoomed view (0.90 – 1.005)
# ─────────────────────────────────────────────────────────────────────────────
fig1 = build_figure(ylim=(0.90, 1.005), ytick_step=0.01)
fname1 = f"{OUT_DIR}/mfdfa_pearson_by_sector.png"
fig1.savefig(fname1, dpi=130, bbox_inches="tight")
plt.close(fig1)
print(f"\nSaved → {fname1}")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 2 — full range (-0.1 – 1.05)
# ─────────────────────────────────────────────────────────────────────────────
fig2 = build_figure(ylim=(-0.1, 1.05), ytick_step=0.10)
fname2 = f"{OUT_DIR}/mfdfa_pearson_by_sector_fullrange.png"
fig2.savefig(fname2, dpi=130, bbox_inches="tight")
plt.close(fig2)
print(f"Saved → {fname2}")


── Sector: Tech ──
  Processing AAPL [#3fb950]... min=0.9700  max=0.9998
  Processing MSFT [#58a6ff]... min=0.9233  max=0.9999
  Processing GOOG [#f85149]... min=0.9692  max=0.9994
  Processing AMZN [#e3b341]... min=0.9744  max=0.9999
  Processing NVDA [#bc8cff]... min=0.9706  max=0.9996
  Processing META [#ff7f0e]... min=0.9287  max=0.9979
  Processing TSLA [#17becf]... min=0.9324  max=0.9995
  Processing NFLX [#f78fb3]... min=0.9841  max=0.9990
  Processing AMD [#d4a017]... min=0.9871  max=0.9999
  Processing INTC [#a8dadc]... min=0.9314  max=0.9997

── Sector: Financials ──
  Processing JPM [#3fb950]... min=0.9842  max=0.9995
  Processing BAC [#58a6ff]... min=0.9786  max=0.9998
  Processing GS [#f85149]... min=0.9468  max=0.9998
  Processing MS [#e3b341]... min=0.9570  max=0.9996
  Processing WFC [#bc8cff]... min=0.9600  max=0.9989
  Processing BLK [#ff7f0e]... min=0.8859  max=1.0000
  Processing C [#17becf]... min=0.9583  max=0.9990
  Processing AXP [#f78fb3]... min=0.9754  max=0.

In [6]:
"""
Detrending Visualisation  —  MF-DFA and MF-DMA variants
=========================================================
Reproduces the style of Figures 6 & 7 in Jiang et al.:
  Left panel  : profile Y(i) (black) + local trend Ỹ(i) (red)
                blue = data points excluded from F calculation
  Right panel : residual ε(i) = Y(i) − Ỹ(i)  +  zero line (red)

One figure (2×2 layout) per method, showing:
  Row 1 : left-to-right sweep  (last s points excluded, shown blue)
  Row 2 : right-to-left sweep  (first s points excluded, shown blue)

Methods: MF-DFA, MF-BDMA (θ=0), MF-CDMA (θ=0.5), MF-FDMA (θ=1)

FIX: Both DFA and DMA operate on the cumulative profile Y(i), not raw
     returns. This matches Eqs. (116)-(118) and Figs. 6-7 in Jiang et al.
"""

import numpy as np
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os, warnings

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# USER PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
TICKER    = "AAPL"
START     = "2010-01-01"
END       = "2026-01-01"
SCALE     = 150          # s  — timescale shown in figure
POLY_ORD  = 1            # DFA polynomial order (1 = linear detrending)
N_SHOW    = 500          # number of profile points to display (like paper)
OUT_DIR   = "detrend_plots"
os.makedirs(OUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────────────────────────────────────
data  = yf.download(TICKER, start=START, end=END, progress=False)
close = data["Close"].dropna().values.ravel().astype(float)
close = close[close > 0]
ret   = np.diff(np.log(close))

# Build profile  Y(i) = cumsum(x - mean(x))   [Eq. 117]
x = ret
Y = np.cumsum(x - x.mean())

# Take a contiguous window of N_SHOW points from the middle of the series
mid    = len(Y) // 2
Y_show = Y[mid : mid + N_SHOW]
N      = len(Y_show)

# ─────────────────────────────────────────────────────────────────────────────
# DETRENDING FUNCTIONS  — both operate on the PROFILE Y
# ─────────────────────────────────────────────────────────────────────────────
s = SCALE

def dfa_trend(Y, start, s, poly_ord=1):
    """
    Polynomial fit of order poly_ord over profile segment [start, start+s).
    Implements Eq. (116): X̃(i) = sum_k a_k * i^k
    """
    seg  = Y[start : start + s]
    t    = np.arange(s)
    coef = np.polyfit(t, seg, poly_ord)
    return np.polyval(coef, t)


def dma_trend(Y, start, s, theta):
    """
    Moving-average trend for DMA computed on the PROFILE Y.  [Eq. (118)]

    For each point k inside the segment, the moving average window uses
        k_back = floor((s-1) * theta)     points into the past
        k_fwd  = ceil ((s-1) * (1-theta)) points into the future
    all indices clipped to the segment boundary.

    theta=0.0  →  backward MA  (MF-BDMA)
    theta=0.5  →  centred  MA  (MF-CDMA)
    theta=1.0  →  forward  MA  (MF-FDMA)
    """
    seg    = Y[start : start + s]
    k_back = int(np.floor((s - 1) * theta))
    k_fwd  = int(np.ceil ((s - 1) * (1.0 - theta)))
    trend  = np.empty(s)
    for k in range(s):
        w_start  = max(0, k - k_back)
        w_end    = min(s, k + k_fwd + 1)
        trend[k] = seg[w_start:w_end].mean()
    return trend


# ─────────────────────────────────────────────────────────────────────────────
# COMPUTE TRENDS + RESIDUALS FOR EVERY SEGMENT  (both sweeps)
# ─────────────────────────────────────────────────────────────────────────────
def compute_trends(Y_show, s, method, theta=None, poly_ord=1):
    """
    Both DFA and DMA operate on the profile Y_show.

    Returns
    -------
    trend_full : array (N,)  — Ỹ(i), NaN where not covered by a full box
    resid_full : array (N,)  — ε(i) = Y(i) − Ỹ(i), NaN likewise
    """
    N  = len(Y_show)
    Ns = N // s

    trend_full = np.full(N, np.nan)
    resid_full = np.full(N, np.nan)

    for v in range(Ns):
        st = v * s
        en = st + s

        if method == "DFA":
            tr = dfa_trend(Y_show, st, s, poly_ord)
        else:
            # FIX: pass profile Y_show, not raw returns
            tr = dma_trend(Y_show, st, s, theta)

        trend_full[st:en] = tr
        resid_full[st:en] = Y_show[st:en] - tr   # ε(i) = Y(i) − Ỹ(i)

    return trend_full, resid_full


i_axis = np.arange(N)

# ─────────────────────────────────────────────────────────────────────────────
# PLOT FUNCTION  (matches paper Figures 6 & 7 style)
# ─────────────────────────────────────────────────────────────────────────────
def make_figure(method, theta, poly_ord=1):
    """
    2×2 figure:
      Row 1: left-to-right  sweep  (last N-used points shown in blue = excluded)
      Row 2: right-to-left  sweep  (first N-used points shown in blue = excluded)

    Both rows plot the PROFILE Y(i) in the left column — consistent with
    Figures 6 and 7 in Jiang et al. regardless of method.
    """
    Ns   = N // s
    used = Ns * s          # number of profile points covered by full boxes

    # ── left-to-right ────────────────────────────────────────────────────────
    trend_lr, resid_lr = compute_trends(Y_show, s, method, theta, poly_ord)

    # ── right-to-left  (flip, compute, flip back) ────────────────────────────
    Y_flip = Y_show[::-1].copy()
    trend_rl_flip, resid_rl_flip = compute_trends(Y_flip, s, method, theta, poly_ord)
    trend_rl = trend_rl_flip[::-1]
    resid_rl = resid_rl_flip[::-1]

    theta_str = f"$\\theta={theta}$" if theta is not None else ""
    title_str = (f"Detrending visualisation  ·  {method}  {theta_str}"
                 f"  ·  {TICKER}  ·  s = {s}")

    fig, axs = plt.subplots(2, 2, figsize=(13, 8), sharey="col")
    fig.suptitle(title_str, fontsize=12, fontweight="bold", y=1.01)

    for row, (trend, resid, label_row) in enumerate([
        (trend_lr, resid_lr, "Left → Right"),
        (trend_rl, resid_rl, "Right → Left"),
    ]):
        # ── left column: profile Y(i) + trend Ỹ(i) ───────────────────────
        ax = axs[row, 0]

        # full profile in black
        ax.plot(i_axis, Y_show, color="black", lw=0.7, zorder=2, label="$Y(i)$")

        # excluded region in blue
        if row == 0:
            ex = slice(used, N)      # LR: tail excluded
        else:
            ex = slice(0, N - used)  # RL: head excluded
        ax.plot(i_axis[ex], Y_show[ex], color="#4488ff", lw=1.0, zorder=3)

        # trend in red
        ax.plot(i_axis, trend, color="#c0392b", lw=1.8, zorder=4,
                label="$\\tilde{Y}(i)$")

        ax.set_xlabel("$i$", fontsize=10)
        ax.set_ylabel("$Y(i)$,  $\\tilde{Y}(i)$", fontsize=9)
        ax.set_title(f"({'ac'[row]}) {label_row} — profile & trend", fontsize=10)
        ax.legend(fontsize=8, loc="best")
        ax.tick_params(direction="in", which="both", top=True, right=True)

        # ── right column: residual ε(i) ───────────────────────────────────
        ax = axs[row, 1]
        ax.plot(i_axis, resid, color="black", lw=0.7)
        ax.axhline(0, color="#c0392b", lw=1.2, zorder=3)
        ax.set_xlabel("$i$", fontsize=10)
        ax.set_ylabel("$\\epsilon(i)$", fontsize=10)
        ax.set_title(f"({'bd'[row]}) {label_row} — residual $\\epsilon(i)$",
                     fontsize=10)
        ax.tick_params(direction="in", which="both", top=True, right=True)

        # dotted vertical lines at segment boundaries
        for v in range(1, Ns):
            for col in range(2):
                axs[row, col].axvline(v * s, color="grey", lw=0.6,
                                      ls=":", alpha=0.7)

    plt.tight_layout()
    safe = method.replace("-", "_")
    if theta is not None:
        safe += f"_theta{str(theta).replace('.', 'p')}"
    fname = f"{OUT_DIR}/{TICKER}_{safe}_s{s}.png"
    plt.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# GENERATE ALL FIGURES
# ─────────────────────────────────────────────────────────────────────────────
VARIANTS = [
    ("DFA",  None),
    ("BDMA", 0.0),
    ("CDMA", 0.5),
    ("FDMA", 1.0),
]

for method, theta in VARIANTS:
    print(f"Plotting {method} ...")
    make_figure(method, theta, poly_ord=POLY_ORD)

print("\nAll done. Figures saved in:", OUT_DIR)

Plotting DFA ...
  saved → detrend_plots/AAPL_DFA_s150.png
Plotting BDMA ...
  saved → detrend_plots/AAPL_BDMA_theta0p0_s150.png
Plotting CDMA ...
  saved → detrend_plots/AAPL_CDMA_theta0p5_s150.png
Plotting FDMA ...
  saved → detrend_plots/AAPL_FDMA_theta1p0_s150.png

All done. Figures saved in: detrend_plots


In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.signal import savgol_filter
from numba import njit

# ─────────────────────────────────────────────────────────────────────────────
# 1. 2008 Configuration  (UNCHANGED from original)
# ─────────────────────────────────────────────────────────────────────────────
tickers              = ["^GSPC", "JPM", "XLF", "C", "GE"]
START_DATA           = "2007-01-01"
END_DATA             = "2009-06-01"
TARGET_DATE          = "2008-09-28"
window_start_offsets = [100, 80, 60, 30]
lookback_offset      = 15
plot_duration        = 150

# ─────────────────────────────────────────────────────────────────────────────
# 2. Mathematical Core  (UNCHANGED from original)
# ─────────────────────────────────────────────────────────────────────────────
@njit
def solve_lppl(tc, m, w, t, p):
    tau    = np.maximum(tc - t, 1e-4)
    log_tau = np.log(tau)
    f, g, h = tau**m, tau**m * np.cos(w * log_tau), tau**m * np.sin(w * log_tau)
    X       = np.ones((len(t), 4))
    X[:, 1], X[:, 2], X[:, 3] = f, g, h
    sol, _, _, _ = np.linalg.lstsq(X, p)
    return sol, f, g, h

@njit
def cost_fn(params, t, p):
    tc, m, w = params
    if tc <= t[-1] or tc > t[-1] + 30: return 1e22
    if m < 0.05 or m > 0.45:           return 1e22
    if w < 5.0  or w > 15.0:           return 1e22
    lin, f, g, h = solve_lppl(tc, m, w, t, p)
    if lin[1] >= 0:                     return 1e22
    osc_amp = np.sqrt(lin[2]**2 + lin[3]**2)
    if osc_amp > abs(lin[1]):           return 1e22
    y_hat = lin[0] + lin[1]*f + lin[2]*g + lin[3]*h
    mse   = np.sum((y_hat - p)**2) / len(t)
    dist_penalty = (tc - t[-1]) / 5.0
    return mse * (1.0 + dist_penalty)

def fit_sornette_model(t, p):
    tc_start = t[-1] + 1
    best_c   = 1e30
    best_init = [tc_start + 5, 0.2, 8.0]
    tcs = np.linspace(tc_start + 2, tc_start + 30, 5)
    ms  = [0.1, 0.3, 0.5]
    ws  = [6.0, 10.0, 14.0]
    for tc_g in tcs:
        for m_g in ms:
            for w_g in ws:
                c = cost_fn(np.array([tc_g, m_g, w_g]), t, p)
                if c < best_c:
                    best_c    = c
                    best_init = [tc_g, m_g, w_g]
    res = minimize(cost_fn, best_init, args=(t, p), method='Nelder-Mead', tol=1e-9)
    return res.x

# ─────────────────────────────────────────────────────────────────────────────
# 3. MF-DFA on arbitrary 1-D series  (from your MF-DFA script, fast path)
# ─────────────────────────────────────────────────────────────────────────────
Q_VALS    = np.arange(-10, 12, 2, dtype=float)
SCALE_MIN = 4
N_SCALES  = 20
POLY_ORD  = 1
OVERLAP   = 0.75

def _overlap_starts(N, s, overlap):
    step = max(1, int(s * (1.0 - overlap)))
    return np.arange(0, N - s + 1, step)

def mfdfa(x, q_vals=Q_VALS, scale_min=SCALE_MIN, n_scales=N_SCALES,
          poly_ord=POLY_ORD, overlap=OVERLAP):
    y         = np.cumsum(x - x.mean())
    N         = len(y)
    scale_max = max(scale_min + 1, N // 4)

    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales).astype(int)
    )
    scales = scales[scales >= scale_min]

    Fqs = np.full((len(q_vals), len(scales)), np.nan)

    for i, s in enumerate(scales):
        starts = _overlap_starts(N, s, overlap)
        if len(starts) < 4:
            continue
        t_seg = np.arange(s)
        rms   = []
        for st in starts:
            seg  = y[st : st + s]
            coef = np.polyfit(t_seg, seg, poly_ord)
            r    = np.sqrt(np.mean((seg - np.polyval(coef, t_seg))**2))
            if r > 0:
                rms.append(r)
        rms = np.asarray(rms)
        if len(rms) == 0:
            continue
        for j, q in enumerate(q_vals):
            if q == 0:
                Fqs[j, i] = np.exp(0.5 * np.mean(np.log(rms**2)))
            else:
                Fqs[j, i] = (np.mean(rms**q))**(1.0 / q)

    hq = np.full(len(q_vals), np.nan)
    for j in range(len(q_vals)):
        row  = Fqs[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() >= 4:
            hq[j] = np.polyfit(np.log(scales[mask]), np.log(row[mask]), 1)[0]

    return scales, Fqs, hq

def multifractal_spectrum(q_vals, hq):
    tau_q = q_vals * hq - 1.0
    wl    = min(5, len(tau_q) if len(tau_q) % 2 == 1 else len(tau_q) - 1)
    tau_s = savgol_filter(tau_q, window_length=wl, polyorder=2) if len(tau_q) >= 5 else tau_q
    alpha   = np.gradient(tau_s, q_vals)
    D_alpha = q_vals * alpha - tau_s
    return tau_q, alpha, D_alpha

# ─────────────────────────────────────────────────────────────────────────────
# 4. 6-panel MF-DFA plot  (adapted from your original)
# ─────────────────────────────────────────────────────────────────────────────
PAPER_COLORS = [
    "#d62728", "#2ca02c", "#1f77b4", "#000000", "#9467bd",
    "#ff7f0e", "#8c564b", "#e377c2", "#17becf", "#bcbd22", "#7f7f7f",
]

def plot_mfdfa_6panel(ticker, window_label, series_label, raw_series,
                      scales, Fqs, hq, q_vals, tau_q, alpha, D_alpha):
    valid   = np.isfinite(hq)
    colors  = (PAPER_COLORS * 4)[: len(q_vals)]

    fig, axs = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle(
        f"MF-DFA on LPPL series  ·  {ticker}  ·  Window {window_label}d  ·  {series_label}",
        fontsize=13, fontweight="bold", y=1.01,
    )

    # (a) raw series
    ax = axs[0, 0]
    ax.plot(raw_series, color="#c0392b", linewidth=0.85)
    ax.set_xlabel("$t$ (trading days)", fontsize=11)
    ax.set_ylabel(series_label, fontsize=11)
    ax.set_title("(a) Input series", fontsize=11)

    # (b) F_q(s) log-log
    ax = axs[0, 1]
    legend_handles = []
    for idx_q, (q, c) in enumerate(zip(q_vals, colors)):
        row  = Fqs[idx_q]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() < 2:
            continue
        h, = ax.loglog(scales[mask], row[mask], "o-",
                       color=c, markersize=3.5, linewidth=1.0)
        legend_handles.append((h, f"$q={int(q)}$"))
    ax.set_xlabel("$s$", fontsize=11)
    ax.set_ylabel("$F_q(s)$", fontsize=11)
    ax.set_title("(b) Fluctuation functions", fontsize=11)
    step  = max(1, len(legend_handles) // 8)
    shown = legend_handles[::step]
    ax.legend([h for h, _ in shown], [l for _, l in shown],
              fontsize=7, ncol=1, loc="best")

    # (c) H(q)
    ax = axs[0, 2]
    ax.plot(q_vals[valid], hq[valid], "-", color="#1f4e8c", linewidth=2)
    ax.set_xlabel("$q$", fontsize=11)
    ax.set_ylabel("$H(q)$", fontsize=11)
    ax.set_title("(c) Generalised Hurst exponent $H(q)$", fontsize=11)
    ax.axhline(0.5, color="grey", lw=0.8, ls="--", alpha=0.6)

    # (d) tau(q)
    ax = axs[1, 0]
    ax.plot(q_vals[valid], tau_q[valid], "-", color="#1f4e8c", linewidth=2)
    ax.set_xlabel("$q$", fontsize=11)
    ax.set_ylabel(r"$\tau(q)$", fontsize=11)
    ax.set_title(r"(d) Scaling exponent $\tau(q)$", fontsize=11)
    ax.axhline(0, color="grey", lw=0.8, ls="--", alpha=0.6)

    # (e) alpha(q)
    ax = axs[1, 1]
    valid_s = np.isfinite(alpha) & np.isfinite(D_alpha)
    ax.plot(q_vals[valid_s], alpha[valid_s], "-", color="#1f4e8c", linewidth=2)
    ax.set_xlabel("$q$", fontsize=11)
    ax.set_ylabel(r"$\alpha(q)$", fontsize=11)
    ax.set_title(r"(e) Singularity strength $\alpha(q)$", fontsize=11)

    # (f) f(alpha)
    ax = axs[1, 2]
    ax.plot(alpha[valid_s], D_alpha[valid_s], "-", color="#1f4e8c", linewidth=2)
    q0_idx = np.argmin(np.abs(q_vals))
    if np.isfinite(alpha[q0_idx]) and np.isfinite(D_alpha[q0_idx]):
        ax.plot(alpha[q0_idx], D_alpha[q0_idx], "o",
                color="#c0392b", markersize=6, zorder=5)
    ax.set_xlabel(r"$\alpha$", fontsize=11)
    ax.set_ylabel(r"$f(\alpha)$", fontsize=11)
    ax.set_title(r"(f) Multifractal spectrum $f(\alpha)$", fontsize=11)

    for ax in axs.ravel():
        ax.tick_params(direction="in", which="both", top=True, right=True)
        ax.spines[["top", "right"]].set_visible(True)

    if valid_s.sum() >= 2:
        delta_alpha = alpha[valid_s].max() - alpha[valid_s].min()
        fig.text(0.5, -0.01,
                 fr"Multifractal width  $\Delta\alpha = {delta_alpha:.4f}$",
                 ha="center", fontsize=10, color="grey")

    plt.tight_layout()
    return fig

# ─────────────────────────────────────────────────────────────────────────────
# 5. Execution  (LPPL loop unchanged; MF-DFA appended after each fit)
# ─────────────────────────────────────────────────────────────────────────────
import os
OUT_DIR = "lppl_mfdfa_plots"
os.makedirs(OUT_DIR, exist_ok=True)

data = yf.download(tickers, start=START_DATA, end=END_DATA, progress=False, auto_adjust=True)

for ticker in tickers:
    prices = data['Close'][ticker].dropna() if len(tickers) > 1 else data['Close'].dropna()
    df     = prices.to_frame(name='Close').reset_index()

    try:
        end_idx = df[df['Date'] <= TARGET_DATE].index[-1]
    except:
        continue

    as_of_date       = df["Date"].iloc[end_idx]
    tc_trading_deltas = []

    print(f"\n{'='*20} 2008 REGIME SCAN: {ticker} {'='*20}")
    print(f"Pivot Date: {as_of_date.date()}")
    print(f"{'Window':<12} | {'tc (Calendar Date)':<20} | {'m':<8} | {'w':<8}")
    print("-" * 75)

    # ── original LPPL overview plot (unchanged) ───────────────────────────────
    plt.figure(figsize=(12, 5))
    limit   = min(len(df) - 1, end_idx + 40)
    view_df = df.iloc[max(0, end_idx - plot_duration): limit + 1]
    plt.plot(view_df["Date"], np.log(view_df["Close"]),
             color="black", alpha=0.3, label="Price History")

    for start_offset in window_start_offsets:
        if end_idx - start_offset < 0:
            continue
        df_fit = df.iloc[end_idx - start_offset: end_idx + 1].copy()
        t_f    = np.arange(len(df_fit)).astype(np.float64)
        p_f    = np.log(df_fit["Close"].values).astype(np.float64)

        tc_rel, m, w = fit_sornette_model(t_f, p_f)
        diff_units    = tc_rel - t_f[-1]
        tc_trading_deltas.append(diff_units)

        tc_calendar_date = as_of_date + pd.tseries.offsets.BusinessDay(n=int(round(diff_units)))

        lin_f, f_v, g_v, h_v = solve_lppl(tc_rel, m, w, t_f, p_f)
        y_fit = lin_f[0] + lin_f[1]*f_v + lin_f[2]*g_v + lin_f[3]*h_v
        plt.plot(df_fit["Date"], y_fit, alpha=0.7, label=f"Fit {start_offset}d")

        print(f"{start_offset} days     | {tc_calendar_date.date()}         | {m:.3f}   | {w:.3f}")

        # ── NEW: MF-DFA on the LPPL fit series and residuals ──────────────────
        residuals = p_f - y_fit               # actual log-price minus LPPL fit
        lppl_fit  = y_fit                     # the smooth LPPL curve itself

        for series_arr, series_label in [
            (lppl_fit,  "LPPL Fit"),
            (residuals, "LPPL Residuals"),
        ]:
            if len(series_arr) < 20:          # too short for DFA
                continue

            scales, Fqs, hq              = mfdfa(series_arr)
            tau_q, alpha, D_alpha        = multifractal_spectrum(Q_VALS, hq)
            fig = plot_mfdfa_6panel(
                ticker, start_offset, series_label,
                series_arr, scales, Fqs, hq, Q_VALS,
                tau_q, alpha, D_alpha,
            )
            safe_label = series_label.replace(" ", "_").lower()
            fname = f"{OUT_DIR}/{ticker.replace('^','').replace('=','_')}_{start_offset}d_{safe_label}.png"
            fig.savefig(fname, dpi=120, bbox_inches="tight")
            plt.close(fig)
            valid_s     = np.isfinite(alpha) & np.isfinite(D_alpha)
            delta_alpha = alpha[valid_s].max() - alpha[valid_s].min() if valid_s.sum() >= 2 else np.nan
            hurst_q2    = hq[np.argmin(np.abs(Q_VALS - 2))]
            print(f"        MF-DFA [{series_label}]  H(q=2)={hurst_q2:.3f}  Δα={delta_alpha:.4f}  → {fname}")

    stability_sigma = np.std(tc_trading_deltas)
    plt.axvline(as_of_date, color="blue", ls="--", label="Analysis Point")
    plt.title(f"{ticker} 2008 | Stability (σ): {stability_sigma:.2f}")
    plt.legend(loc='lower left', fontsize='small', ncol=2)
    plt.grid(alpha=0.3)
    overview_fname = f"{OUT_DIR}/{ticker.replace('^','').replace('=','_')}_overview.png"
    plt.savefig(overview_fname, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"  Overview → {overview_fname}")

    print(f"Consensus Stability Index: {stability_sigma:.2f}")
    print("-" * 80)

print(f"\nAll done. Figures saved in: {OUT_DIR}/")


==================== 2008 REGIME SCAN: ^GSPC ====================
Pivot Date: 2008-09-26
Window       | tc (Calendar Date)   | m        | w       
---------------------------------------------------------------------------
100 days     | 2008-10-01         | 0.100   | 6.000
        MF-DFA [LPPL Fit]  H(q=2)=1.664  Δα=2.1187  → lppl_mfdfa_plots/GSPC_100d_lppl_fit.png
        MF-DFA [LPPL Residuals]  H(q=2)=1.041  Δα=1.4137  → lppl_mfdfa_plots/GSPC_100d_lppl_residuals.png
80 days     | 2008-10-01         | 0.100   | 6.000
        MF-DFA [LPPL Fit]  H(q=2)=1.680  Δα=1.9962  → lppl_mfdfa_plots/GSPC_80d_lppl_fit.png
        MF-DFA [LPPL Residuals]  H(q=2)=1.050  Δα=1.0985  → lppl_mfdfa_plots/GSPC_80d_lppl_residuals.png
60 days     | 2008-10-01         | 0.100   | 6.000
        MF-DFA [LPPL Fit]  H(q=2)=1.723  Δα=2.0393  → lppl_mfdfa_plots/GSPC_60d_lppl_fit.png
        MF-DFA [LPPL Residuals]  H(q=2)=0.942  Δα=1.7278  → lppl_mfdfa_plots/GSPC_60d_lppl_residuals.png
30 days     | 2008-10-01  

In [6]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import minimize
from scipy.signal import savgol_filter
from numba import njit
import os, warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# 1. CONFIGURATION  (unchanged from original)
# ─────────────────────────────────────────────────────────────────────────────
tickers              = ["^GSPC", "JPM", "XLF", "C", "GE"]
START_DATA           = "2007-01-01"
END_DATA             = "2009-06-01"
TARGET_DATE          = "2008-09-28"
window_start_offsets = [120 ,100, 80, 60]
lookback_offset      = 15
plot_duration        = 150

# NEW: rolling analysis — how many days back from TARGET_DATE to roll over
ROLL_DAYS            = 60          # roll the pivot across this many days
ROLL_STEP            = 5           # step size in trading days
SURROGATE_N          = 100         # phase-randomisation realisations
MIN_N_DFA            = 60          # minimum series length for reliable MF-DFA
MIN_DECADES          = 0.6         # minimum log10 scale-range (decades)

OUT_DIR = "lppl_mfdfa_v2_plots"
os.makedirs(OUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# 2. LPPL CORE  (completely unchanged)
# ─────────────────────────────────────────────────────────────────────────────
@njit
def solve_lppl(tc, m, w, t, p):
    tau     = np.maximum(tc - t, 1e-4)
    log_tau = np.log(tau)
    f, g, h = tau**m, tau**m * np.cos(w * log_tau), tau**m * np.sin(w * log_tau)
    X       = np.ones((len(t), 4))
    X[:, 1], X[:, 2], X[:, 3] = f, g, h
    sol, _, _, _ = np.linalg.lstsq(X, p)
    return sol, f, g, h

@njit
def cost_fn(params, t, p):
    tc, m, w = params
    if tc <= t[-1] or tc > t[-1] + 30: return 1e22
    if m < 0.05 or m > 0.45:           return 1e22
    if w < 5.0  or w > 15.0:           return 1e22
    lin, f, g, h = solve_lppl(tc, m, w, t, p)
    if lin[1] >= 0:                     return 1e22
    osc_amp = np.sqrt(lin[2]**2 + lin[3]**2)
    if osc_amp > abs(lin[1]):           return 1e22
    y_hat = lin[0] + lin[1]*f + lin[2]*g + lin[3]*h
    mse   = np.sum((y_hat - p)**2) / len(t)
    return mse * (1.0 + (tc - t[-1]) / 5.0)

def fit_sornette_model(t, p):
    tc_start  = t[-1] + 1
    best_c    = 1e30
    best_init = [tc_start + 5, 0.2, 8.0]
    for tc_g in np.linspace(tc_start + 2, tc_start + 30, 5):
        for m_g in [0.1, 0.3, 0.5]:
            for w_g in [6.0, 10.0, 14.0]:
                c = cost_fn(np.array([tc_g, m_g, w_g]), t, p)
                if c < best_c:
                    best_c    = c
                    best_init = [tc_g, m_g, w_g]
    res = minimize(cost_fn, best_init, args=(t, p), method='Nelder-Mead', tol=1e-9)
    return res.x

# ─────────────────────────────────────────────────────────────────────────────
# 3. MF-DFA  (unchanged internals, wrapped with validity guard)
# ─────────────────────────────────────────────────────────────────────────────
Q_VALS   = np.arange(-10, 12, 2, dtype=float)
SCALE_MIN = 2
N_SCALES  = 50
POLY_ORD  = 1
OVERLAP   = 0.75

def _overlap_starts(N, s, overlap):
    step = max(1, int(s * (1.0 - overlap)))
    return np.arange(0, N - s + 1, step)

def mfdfa(x, q_vals=Q_VALS, scale_min=SCALE_MIN, n_scales=N_SCALES,
          poly_ord=POLY_ORD, overlap=OVERLAP):
    y         = np.cumsum(x - x.mean())
    N         = len(y)
    scale_max = max(scale_min + 1, N // 4)
    scales    = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales).astype(int))
    scales    = scales[scales >= scale_min]
    Fqs       = np.full((len(q_vals), len(scales)), np.nan)

    for i, s in enumerate(scales):
        starts = _overlap_starts(N, s, overlap)
        if len(starts) < 4: continue
        t_seg  = np.arange(s)
        rms    = []
        for st in starts:
            seg  = y[st: st + s]
            coef = np.polyfit(t_seg, seg, poly_ord)
            r    = np.sqrt(np.mean((seg - np.polyval(coef, t_seg))**2))
            if r > 0: rms.append(r)
        rms = np.asarray(rms)
        if len(rms) == 0: continue
        for j, q in enumerate(q_vals):
            if q == 0:
                Fqs[j, i] = np.exp(0.5 * np.mean(np.log(rms**2)))
            else:
                Fqs[j, i] = (np.mean(rms**q))**(1.0 / q)

    hq = np.full(len(q_vals), np.nan)
    for j in range(len(q_vals)):
        row  = Fqs[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() >= 4:
            hq[j] = np.polyfit(np.log(scales[mask]), np.log(row[mask]), 1)[0]
    return scales, Fqs, hq

def multifractal_spectrum(hq):
    tau_q = Q_VALS * hq - 1.0
    wl    = min(5, len(tau_q) if len(tau_q) % 2 == 1 else len(tau_q) - 1)
    tau_s = savgol_filter(tau_q, wl, 2) if len(tau_q) >= 5 else tau_q
    alpha   = np.gradient(tau_s, Q_VALS)
    D_alpha = Q_VALS * alpha - tau_s
    return tau_q, alpha, D_alpha

def delta_alpha(x):
    """Return Δα for series x, or NaN if validity checks fail."""
    N = len(x)
    if N < MIN_N_DFA:
        return np.nan, "too_short"
    scale_max = N // 4
    decades   = np.log10(scale_max) - np.log10(SCALE_MIN)
    if decades < MIN_DECADES:
        return np.nan, "too_few_decades"
    _, _, hq        = mfdfa(x)
    _, alpha, D_alp = multifractal_spectrum(hq)
    vs              = np.isfinite(alpha) & np.isfinite(D_alp)
    if vs.sum() < 4:
        return np.nan, "unstable_spectrum"
    da = alpha[vs].max() - alpha[vs].min()
    return da, "ok"

# ─────────────────────────────────────────────────────────────────────────────
# 4. SURROGATE TEST
#    Phase-randomise x (preserves power spectrum, destroys nonlinear deps).
#    Return p-value: fraction of surrogates with Δα >= observed Δα.
# ─────────────────────────────────────────────────────────────────────────────
def surrogate_test(x, n=SURROGATE_N):
    obs_da, flag = delta_alpha(x)
    if flag != "ok":
        return obs_da, np.nan, np.nan, flag

    rng      = np.random.default_rng(42)
    sur_das  = []
    fft_x    = np.fft.rfft(x)
    amps     = np.abs(fft_x)

    for _ in range(n):
        phases    = rng.uniform(0, 2 * np.pi, len(fft_x))
        # keep DC and Nyquist phases at 0 to preserve mean/symmetry
        phases[0] = 0
        if len(x) % 2 == 0:
            phases[-1] = 0
        sur_fft   = amps * np.exp(1j * phases)
        sur       = np.fft.irfft(sur_fft, n=len(x))
        da_s, _   = delta_alpha(sur)
        if np.isfinite(da_s):
            sur_das.append(da_s)

    sur_das = np.array(sur_das)
    if len(sur_das) == 0:
        return obs_da, np.nan, np.nan, "surrogate_failed"

    p_val    = np.mean(sur_das >= obs_da)   # fraction of surrogates >= observed
    sur_mean = sur_das.mean()
    return obs_da, p_val, sur_mean, "ok"

# ─────────────────────────────────────────────────────────────────────────────
# 5. PLOTS
# ─────────────────────────────────────────────────────────────────────────────
PAPER_COLORS = ["#d62728","#2ca02c","#1f77b4","#000000","#9467bd",
                "#ff7f0e","#8c564b","#e377c2","#17becf","#bcbd22","#7f7f7f"]

def plot_mfdfa_6panel(ticker, window_label, series_label, raw_series,
                      scales, Fqs, hq, tau_q, alpha, D_alpha,
                      obs_da=None, p_val=None, sur_mean=None):
    valid  = np.isfinite(hq)
    colors = (PAPER_COLORS * 4)[:len(Q_VALS)]
    fig, axs = plt.subplots(2, 3, figsize=(15, 9))

    sig_str = ""
    if p_val is not None and np.isfinite(p_val):
        sig_str = f"  |  surrogate p={p_val:.3f}  (μ_sur={sur_mean:.3f})"

    fig.suptitle(
        f"MF-DFA · {ticker} · {window_label}d · {series_label}{sig_str}",
        fontsize=12, fontweight="bold", y=1.01)

    ax = axs[0, 0]
    ax.plot(raw_series, color="#c0392b", lw=0.85)
    ax.set_xlabel("$t$ (trading days)"); ax.set_ylabel(series_label)
    ax.set_title("(a) Input series")

    ax = axs[0, 1]
    handles = []
    for idx_q, (q, c) in enumerate(zip(Q_VALS, colors)):
        row  = Fqs[idx_q]; mask = np.isfinite(row) & (row > 0)
        if mask.sum() < 2: continue
        h, = ax.loglog(scales[mask], row[mask], "o-", color=c, markersize=3.5, lw=1.0)
        handles.append((h, f"$q={int(q)}$"))
    ax.set_xlabel("$s$"); ax.set_ylabel("$F_q(s)$")
    ax.set_title("(b) Fluctuation functions")
    step  = max(1, len(handles) // 8); shown = handles[::step]
    ax.legend([h for h, _ in shown], [l for _, l in shown], fontsize=7, ncol=1)

    ax = axs[0, 2]
    ax.plot(Q_VALS[valid], hq[valid], "-", color="#1f4e8c", lw=2)
    ax.axhline(0.5, color="grey", lw=0.8, ls="--", alpha=0.6)
    ax.set_xlabel("$q$"); ax.set_ylabel("$H(q)$")
    ax.set_title("(c) Generalised Hurst $H(q)$")

    ax = axs[1, 0]
    ax.plot(Q_VALS[valid], tau_q[valid], "-", color="#1f4e8c", lw=2)
    ax.axhline(0, color="grey", lw=0.8, ls="--", alpha=0.6)
    ax.set_xlabel("$q$"); ax.set_ylabel(r"$\tau(q)$")
    ax.set_title(r"(d) Scaling exponent $\tau(q)$")

    vs = np.isfinite(alpha) & np.isfinite(D_alpha)
    ax = axs[1, 1]
    ax.plot(Q_VALS[vs], alpha[vs], "-", color="#1f4e8c", lw=2)
    ax.set_xlabel("$q$"); ax.set_ylabel(r"$\alpha(q)$")
    ax.set_title(r"(e) Singularity strength $\alpha(q)$")

    ax = axs[1, 2]
    ax.plot(alpha[vs], D_alpha[vs], "-", color="#1f4e8c", lw=2)
    q0 = np.argmin(np.abs(Q_VALS))
    if np.isfinite(alpha[q0]) and np.isfinite(D_alpha[q0]):
        ax.plot(alpha[q0], D_alpha[q0], "o", color="#c0392b", markersize=6, zorder=5)
    ax.set_xlabel(r"$\alpha$"); ax.set_ylabel(r"$f(\alpha)$")
    ax.set_title(r"(f) Multifractal spectrum $f(\alpha)$")

    for ax in axs.ravel():
        ax.tick_params(direction="in", which="both", top=True, right=True)
        ax.spines[["top","right"]].set_visible(True)

    footer = fr"$\Delta\alpha = {obs_da:.4f}$" if (obs_da and np.isfinite(obs_da)) else ""
    if footer:
        fig.text(0.5, -0.01, footer, ha="center", fontsize=10, color="grey")

    plt.tight_layout()
    return fig


def plot_rolling_trajectory(ticker, dates, da_by_window, tc_sigma_series,
                            joint_score_series, target_date):
    """
    3-row plot:
      Row 1: Δα trajectory per window (lines)
      Row 2: tc stability σ (rolling)
      Row 3: Joint crash score
    """
    fig = plt.figure(figsize=(14, 10))
    gs  = gridspec.GridSpec(3, 1, hspace=0.45)

    ax1 = fig.add_subplot(gs[0])
    colors_w = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
    for (wlab, da_series), col in zip(da_by_window.items(), colors_w):
        arr = np.array(da_series, dtype=float)
        ax1.plot(dates, arr, "o-", color=col, markersize=3, lw=1.2,
                 label=f"{wlab}d window")
    ax1.axvline(target_date, color="black", ls="--", lw=1, label="Target date")
    ax1.set_ylabel(r"$\Delta\alpha$ (residuals)", fontsize=11)
    ax1.set_title(f"{ticker}  —  Rolling Δα trajectory (LPPL residuals)", fontsize=12)
    ax1.legend(fontsize=8, ncol=2); ax1.grid(alpha=0.3)

    ax2 = fig.add_subplot(gs[1])
    tc_arr = np.array(tc_sigma_series, dtype=float)
    ax2.plot(dates, tc_arr, "s-", color="#9467bd", markersize=3, lw=1.2)
    ax2.axvline(target_date, color="black", ls="--", lw=1)
    ax2.set_ylabel(r"$\sigma(t_c)$ across windows", fontsize=11)
    ax2.set_title("tc Stability index  (lower = more consensus)", fontsize=12)
    ax2.grid(alpha=0.3)

    ax3 = fig.add_subplot(gs[2])
    js_arr = np.array(joint_score_series, dtype=float)
    # normalise 0-1 for display
    valid_js = js_arr[np.isfinite(js_arr)]
    if len(valid_js) > 0:
        js_norm = (js_arr - valid_js.min()) / (valid_js.ptp() + 1e-12)
    else:
        js_norm = js_arr
    ax3.fill_between(dates, js_norm, alpha=0.35, color="#d62728")
    ax3.plot(dates, js_norm, "-", color="#d62728", lw=1.5)
    ax3.axvline(target_date, color="black", ls="--", lw=1)
    ax3.set_ylabel("Joint score (normalised)", fontsize=11)
    ax3.set_title(r"Joint Crash Score  =  $\Delta\alpha_{mean}$ × $(1 / \sigma_{t_c})$"
                  "  (higher = stronger signal)", fontsize=12)
    ax3.set_ylim(0, 1.05); ax3.grid(alpha=0.3)

    for ax in [ax1, ax2, ax3]:
        ax.set_xlabel("Analysis date")

    plt.suptitle(f"{ticker}  ·  Rolling LPPL + MF-DFA joint signal",
                 fontsize=13, fontweight="bold")
    return fig

# ─────────────────────────────────────────────────────────────────────────────
# 6. MAIN EXECUTION
# ─────────────────────────────────────────────────────────────────────────────
data = yf.download(tickers, start=START_DATA, end=END_DATA,
                   progress=False, auto_adjust=True)

for ticker in tickers:
    prices = data['Close'][ticker].dropna() if len(tickers) > 1 else data['Close'].dropna()
    df     = prices.to_frame(name='Close').reset_index()

    try:
        anchor_idx = df[df['Date'] <= TARGET_DATE].index[-1]
    except:
        continue

    print(f"\n{'='*22} {ticker} {'='*22}")

    # ── containers for rolling analysis ──────────────────────────────────────
    roll_dates       = []
    da_by_window     = {w: [] for w in window_start_offsets}
    tc_sigma_series  = []
    joint_score_series = []

    # ── rolling pivot loop ───────────────────────────────────────────────────
    pivot_offsets = range(ROLL_DAYS, -1, -ROLL_STEP)   # ROLL_DAYS ago -> TARGET_DATE

    for days_back in pivot_offsets:
        end_idx = anchor_idx - days_back
        if end_idx < max(window_start_offsets):
            continue

        as_of_date = df["Date"].iloc[end_idx]
        roll_dates.append(as_of_date)

        tc_deltas   = []
        da_this_row = {}

        for start_offset in window_start_offsets:
            if end_idx - start_offset < 0:
                da_by_window[start_offset].append(np.nan)
                continue

            df_fit = df.iloc[end_idx - start_offset: end_idx + 1].copy()
            t_f    = np.arange(len(df_fit)).astype(np.float64)
            p_f    = np.log(df_fit["Close"].values).astype(np.float64)

            tc_rel, m, w = fit_sornette_model(t_f, p_f)
            tc_deltas.append(tc_rel - t_f[-1])

            lin_f, f_v, g_v, h_v = solve_lppl(tc_rel, m, w, t_f, p_f)
            y_fit     = lin_f[0] + lin_f[1]*f_v + lin_f[2]*g_v + lin_f[3]*h_v
            residuals = p_f - y_fit

            da, flag = delta_alpha(residuals)
            da_by_window[start_offset].append(da)
            da_this_row[start_offset] = da

            # ── full 6-panel + surrogate only at TARGET_DATE (anchor) ────────
            if days_back == 0:
                # surrogate test
                obs_da, p_val, sur_mean, sflag = surrogate_test(residuals)

                # validity guard: skip LPPL Fit panel (it's deterministic)
                _, _, hq          = mfdfa(residuals)
                tau_q, alpha, Da  = multifractal_spectrum(hq)
                sc, Fqs_r, _      = mfdfa(residuals)

                fig = plot_mfdfa_6panel(
                    ticker, start_offset, "LPPL Residuals",
                    residuals, sc, Fqs_r, hq, tau_q, alpha, Da,
                    obs_da=obs_da, p_val=p_val, sur_mean=sur_mean)
                fname = (f"{OUT_DIR}/{ticker.replace('^','').replace('=','_')}"
                         f"_{start_offset}d_residuals_surrogate.png")
                fig.savefig(fname, dpi=120, bbox_inches="tight")
                plt.close(fig)

                sig = "✓ GENUINE" if (np.isfinite(p_val) and p_val < 0.05) else "✗ linear/artefact"
                print(f"  {start_offset}d | Δα={obs_da:.3f} | p={p_val:.3f} "
                      f"| μ_sur={sur_mean:.3f} | {sig} | flag={sflag}")

        # tc stability
        sigma_tc = np.std(tc_deltas) if len(tc_deltas) >= 2 else np.nan
        tc_sigma_series.append(sigma_tc)

        # joint score: mean(Δα) × (1 / σ_tc)   — higher when Δα is large AND tc is stable
        valid_das = [v for v in da_this_row.values() if np.isfinite(v)]
        mean_da   = np.mean(valid_das) if valid_das else np.nan
        if np.isfinite(mean_da) and np.isfinite(sigma_tc) and sigma_tc > 0:
            joint_score_series.append(mean_da / sigma_tc)
        else:
            joint_score_series.append(np.nan)

    # ── rolling trajectory plot ───────────────────────────────────────────────
    if len(roll_dates) >= 3:
        fig_roll = plot_rolling_trajectory(
            ticker, roll_dates,
            da_by_window, tc_sigma_series, joint_score_series,
            df["Date"].iloc[anchor_idx])
        rfname = f"{OUT_DIR}/{ticker.replace('^','').replace('=','_')}_rolling_trajectory.png"
        fig_roll.savefig(rfname, dpi=120, bbox_inches="tight")
        plt.close(fig_roll)
        print(f"  Rolling trajectory → {rfname}")

    # ── original LPPL overview plot at TARGET_DATE (unchanged) ───────────────
    end_idx = anchor_idx
    as_of_date = df["Date"].iloc[end_idx]
    tc_trading_deltas = []

    fig_ov, ax_ov = plt.subplots(figsize=(12, 5))
    limit   = min(len(df) - 1, end_idx + 40)
    view_df = df.iloc[max(0, end_idx - plot_duration): limit + 1]
    ax_ov.plot(view_df["Date"], np.log(view_df["Close"]),
               color="black", alpha=0.3, label="Price History")

    print(f"\n  {'Window':<10} | {'tc date':<20} | {'m':<7} | {'w':<7}")
    print("  " + "-"*55)

    for start_offset in window_start_offsets:
        if end_idx - start_offset < 0: continue
        df_fit = df.iloc[end_idx - start_offset: end_idx + 1].copy()
        t_f    = np.arange(len(df_fit)).astype(np.float64)
        p_f    = np.log(df_fit["Close"].values).astype(np.float64)
        tc_rel, m, w = fit_sornette_model(t_f, p_f)
        tc_trading_deltas.append(tc_rel - t_f[-1])
        tc_cal = as_of_date + pd.tseries.offsets.BusinessDay(n=int(round(tc_rel - t_f[-1])))
        lin_f, f_v, g_v, h_v = solve_lppl(tc_rel, m, w, t_f, p_f)
        y_fit = lin_f[0] + lin_f[1]*f_v + lin_f[2]*g_v + lin_f[3]*h_v
        ax_ov.plot(df_fit["Date"], y_fit, alpha=0.7, label=f"Fit {start_offset}d")
        print(f"  {start_offset:<10} | {tc_cal.date()}         | {m:.3f}   | {w:.3f}")

    sigma = np.std(tc_trading_deltas)
    ax_ov.axvline(as_of_date, color="blue", ls="--", label="Analysis Point")
    ax_ov.set_title(f"{ticker} 2008 | σ(tc)={sigma:.2f}")
    ax_ov.legend(loc='lower left', fontsize='small', ncol=2)
    ax_ov.grid(alpha=0.3)
    ov_fname = f"{OUT_DIR}/{ticker.replace('^','').replace('=','_')}_overview.png"
    fig_ov.savefig(ov_fname, dpi=120, bbox_inches="tight")
    plt.close(fig_ov)
    print(f"  Overview → {ov_fname}  |  σ(tc)={sigma:.2f}")
    print("  " + "="*55)

print(f"\nAll done. Figures in: {OUT_DIR}/")


====================== ^GSPC ======================
  120d | Δα=1.573 | p=0.490 | μ_sur=1.544 | ✗ linear/artefact | flag=ok
  100d | Δα=1.616 | p=0.510 | μ_sur=1.613 | ✗ linear/artefact | flag=ok
  80d | Δα=1.253 | p=0.920 | μ_sur=1.729 | ✗ linear/artefact | flag=ok
  60d | Δα=2.037 | p=0.570 | μ_sur=2.112 | ✗ linear/artefact | flag=ok
  Rolling trajectory → lppl_mfdfa_v2_plots/GSPC_rolling_trajectory.png

  Window     | tc date              | m       | w      
  -------------------------------------------------------
  120        | 2008-10-01         | 0.100   | 6.000
  100        | 2008-10-01         | 0.100   | 6.000
  80         | 2008-10-01         | 0.100   | 6.000
  60         | 2008-10-01         | 0.100   | 6.000
  Overview → lppl_mfdfa_v2_plots/GSPC_overview.png  |  σ(tc)=0.00

====================== JPM ======================
  120d | Δα=1.567 | p=0.460 | μ_sur=1.560 | ✗ linear/artefact | flag=ok
  100d | Δα=1.798 | p=0.430 | μ_sur=1.714 | ✗ linear/artefact | flag=ok
  80d 